# Cleaning Data

## Did you like the movie? - Cleaning the database

### Loading libraries

The libraries we will use for cleaning our dataset are `pandas` (for working with the data set itself), `numpy` (for working with NAs and other array computations) and `re` (so we can use regular expressions for cleaning certain columns).

In [1]:
import pandas as pd
from numpy import nan
import re

### Team 1

The first order of business is loading the dataset into a `pandas DataFrame` object.

In [2]:
team1 = pd.read_csv('../datasets/team1.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../datasets/team1.csv'

Let’s explore our database so we can get a glimpse at the data

In [ ]:
team1.head()

We can check the dimensions of our dataframe with `pd.shape`.

In [4]:
team1.shape

(201, 32)

We observe that there are 201 observations with 32 attributes.

#### Column cleaning

In this section we will clean the column names so they look homogeneous, without extra (invisible) characters.

First of all, we print a list of column names.

In [5]:
team1.columns

Index(['Name ', 'Year', 'Duration (MINUTES)', 'Genre ',
       'Budget (MILLIONS USD)', 'Revenue  (MILLIONS USD)', 'IMDB',
       'METACRITIC', 'ROTTEN TOMATOS', 'FILMAFFINITY', 'Personal op',
       'Average', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10',
       'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20'],
      dtype='object')

We will rename `Name` to `Title`, and correct the capitalization and some spelling mistakes.

In [6]:
team1.rename(columns = {'Name ':'Title', 'Duration (MINUTES)':'Runtime (Minutes)', 'Genre ':'Genre', 'Budget (MILLIONS USD)':'Budget (Millions USD)',
                        'Revenue  (MILLIONS USD)':'Revenue (Millions USD)', 'IMDB':'IMDb', 'METACRITIC':'Metacritic', 'ROTTEN TOMATOS':'Rotten Tomatoes',
                        'FILMAFFINITY':'Filmaffinity', 'Personal op':'Personal'}, inplace=True)
team1.columns

Index(['Title', 'Year', 'Runtime (Minutes)', 'Genre', 'Budget (Millions USD)',
       'Revenue (Millions USD)', 'IMDb', 'Metacritic', 'Rotten Tomatoes',
       'Filmaffinity', 'Personal', 'Average', 'P1', 'P2', 'P3', 'P4', 'P5',
       'P6', 'P7', 'P8', 'P9', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16',
       'P17', 'P18', 'P19', 'P20'],
      dtype='object')

Finally, we will set the `Title` column as the indexing column of our dataframe.

In [7]:
team1.set_index('Title', inplace=True)

#### Data cleaning

We will follow the guidelines stated below for cleaning the data for each column:
- `Year`: This column will be left unchanged. Data type can be `str` or `int`.
- `Runtime`: Data type should be `int`. Remove any indication to a unit of measurement.
- `Genre`: Data type should be `str`. We will capitalize the data, and if there is more than one genre, we will keep the first one and remove the rest.
- `Budget`: Data type can be `int` or `float`. To that end, we will remove dollar signs as well as the thousands separator. We will scale the data so the unit of measurment is millions of dollars.
- `Revenue`: Same treatment as that of `Budget`.
- Rating columns: Data type can be `int` or `float`. Change scale to [0,100].

Let's explore the type of each of the columns. In the table below we can also see if there is missing data.

In [8]:
team1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 201 entries, Me Before you  to Men in black 2
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Year                    201 non-null    int64  
 1   Runtime (Minutes)       201 non-null    int64  
 2   Genre                   201 non-null    object 
 3   Budget (Millions USD)   199 non-null    object 
 4   Revenue (Millions USD)  197 non-null    object 
 5   IMDb                    201 non-null    int64  
 6   Metacritic              200 non-null    float64
 7   Rotten Tomatoes         200 non-null    float64
 8   Filmaffinity            200 non-null    float64
 9   Personal                198 non-null    float64
 10  Average                 201 non-null    float64
 11  P1                      166 non-null    float64
 12  P2                      182 non-null    float64
 13  P3                      166 non-null    float64
 14  P4                     

We observe that columns `Budget`, `Revenue`, and `P9` have the wrong type. Let's explore each column in turn, and convert them to `int` or `float` as appropriate.

In [9]:
team1['Budget (Millions USD)'].unique()

array(['20', '12', '80', '37', '65', '90', '2.4', '78', '55', nan, '70',
       '45', '100', '50', '60', '40', '150', '29', '3.3', '133', '15',
       '81.2', '72', '4', '130', '34', '19.5', '11', '149', '75', '30',
       '52', '35', '17', '6.5', '200', '5', '14', '10', '237', '260',
       '250', '180', '175', '18', '140', '26', '183', '160', '356', '69',
       '22', '115', '92', '28', '7', '6', '9', '178', '63', '4.5', '165',
       '25', '19', '29.5', '13', '54', '2.2', '24', '5.5', '110', '58',
       '125', '150 000 ', '61', '62', '858 000 ', '4.4', '2.8', '8',
       '170', '232', '103', '230', '120', '1', '105', '33'], dtype=object)

We've found the culprit. There are a couple of entries with trailing spaces. Further, those same entries do not appear to be in the correct scale. Let's see to which movies they correspond.

In [10]:
print(team1[team1['Budget (Millions USD)']=='150 000 '].index.values)
print(team1[team1['Budget (Millions USD)']=='858 000 '].index.values)

['Inheritance']
['Bambi']


A quick search on the internet shows that these correspond to the total budget in dollars, not in millions of dollars. So, we can safely divide each of them by $10^6$.

In [11]:
team1.at['Inheritance','Budget (Millions USD)'] = 0.15
team1.at['Bambi','Budget (Millions USD)'] = 0.858

In [12]:
team1['Budget (Millions USD)'] = team1['Budget (Millions USD)'].astype(float)
team1['Budget (Millions USD)'].unique()

array([2.00e+01, 1.20e+01, 8.00e+01, 3.70e+01, 6.50e+01, 9.00e+01,
       2.40e+00, 7.80e+01, 5.50e+01,      nan, 7.00e+01, 4.50e+01,
       1.00e+02, 5.00e+01, 6.00e+01, 4.00e+01, 1.50e+02, 2.90e+01,
       3.30e+00, 1.33e+02, 1.50e+01, 8.12e+01, 7.20e+01, 4.00e+00,
       1.30e+02, 3.40e+01, 1.95e+01, 1.10e+01, 1.49e+02, 7.50e+01,
       3.00e+01, 5.20e+01, 3.50e+01, 1.70e+01, 6.50e+00, 2.00e+02,
       5.00e+00, 1.40e+01, 1.00e+01, 2.37e+02, 2.60e+02, 2.50e+02,
       1.80e+02, 1.75e+02, 1.80e+01, 1.40e+02, 2.60e+01, 1.83e+02,
       1.60e+02, 3.56e+02, 6.90e+01, 2.20e+01, 1.15e+02, 9.20e+01,
       2.80e+01, 7.00e+00, 6.00e+00, 9.00e+00, 1.78e+02, 6.30e+01,
       4.50e+00, 1.65e+02, 2.50e+01, 1.90e+01, 2.95e+01, 1.30e+01,
       5.40e+01, 2.20e+00, 2.40e+01, 5.50e+00, 1.10e+02, 5.80e+01,
       1.25e+02, 1.50e-01, 6.10e+01, 6.20e+01, 8.58e-01, 4.40e+00,
       2.80e+00, 8.00e+00, 1.70e+02, 2.32e+02, 1.03e+02, 2.30e+02,
       1.20e+02, 1.00e+00, 1.05e+02, 3.30e+01])

In [13]:
team1['Revenue (Millions USD)'].unique()

array(['208.44', '56.69', '214.94', '96.45', '262.82', '962.54', '5.22',
       '695.22', '678.22', nan, '299.16', '212.74', '230.88', '171.26',
       '488.62', '84.38', '346', '1671', '115', '49', '300', '97.57',
       '213', '205', '457', '392', '123', '226.94', '195', '263', '821',
       '875', '1,023', '207', '50', '352', '709', '117', '100', '113.1',
       '130.16', '271.45', '320.41', '257.58', '408.49', '1,374.95',
       '100.5', '6.69', '463.4', '62.54', '2,923.70', '592.47', '569.62',
       '759.85', '859.1', '233.5', '141.78', '585.8', '623.93', '449.32',
       '165.33', '1,025.46', '1,054.30', '1.27', '474.96', '2,779.43',
       '543.28', '394.43', '366.08', '579.72', '631.68', '90.45',
       '102.73', '749.2', '0.43', '704.24', '121.61', '92.56', '59.49',
       '270', '39.7', '672.8', '91.5', '370', '153.4', '230', '863.8',
       '294.8', '209.9', '101.2', '84.6', '90', '255.5', '95.6', '476.8',
       '910.8', '701.8', '264.1', '41.6', '106', '388.8', '133', '47

There are a few movies with revenue in billions of dollars, one with revenue less than a million dollars, and a couple with a thousands separator which needs to be removed. Let's fix these issues by using the method `pd.str.replace()`.

In [14]:
team1['Revenue (Millions USD)']=team1['Revenue (Millions USD)'].str.replace(',', '')
team1['Revenue (Millions USD)']=team1['Revenue (Millions USD)'].str.replace(' USD', '')
team1['Revenue (Millions USD)']=team1['Revenue (Millions USD)'].str.replace('B', '')
team1['Revenue (Millions USD)'].unique()

array(['208.44', '56.69', '214.94', '96.45', '262.82', '962.54', '5.22',
       '695.22', '678.22', nan, '299.16', '212.74', '230.88', '171.26',
       '488.62', '84.38', '346', '1671', '115', '49', '300', '97.57',
       '213', '205', '457', '392', '123', '226.94', '195', '263', '821',
       '875', '1023', '207', '50', '352', '709', '117', '100', '113.1',
       '130.16', '271.45', '320.41', '257.58', '408.49', '1374.95',
       '100.5', '6.69', '463.4', '62.54', '2923.70', '592.47', '569.62',
       '759.85', '859.1', '233.5', '141.78', '585.8', '623.93', '449.32',
       '165.33', '1025.46', '1054.30', '1.27', '474.96', '2779.43',
       '543.28', '394.43', '366.08', '579.72', '631.68', '90.45',
       '102.73', '749.2', '0.43', '704.24', '121.61', '92.56', '59.49',
       '270', '39.7', '672.8', '91.5', '370', '153.4', '230', '863.8',
       '294.8', '209.9', '101.2', '84.6', '90', '255.5', '95.6', '476.8',
       '910.8', '701.8', '264.1', '41.6', '106', '388.8', '133', '47.3',
 

In [15]:
print(team1[team1['Revenue (Millions USD)']=='304 000'].index.values)
print(team1[team1['Revenue (Millions USD)']=='1'].index.values)
print(team1[team1['Revenue (Millions USD)']=='1.3'].index.values)

['Inheritance']
['Deadpool and wolverine' 'Harry Potter and the sorcers stone']
['Harry Potter and the deathly hallows I']


In [16]:
team1.at['Inheritance','Revenue (Millions USD)'] = 0.304
team1.at['Deadpool and wolverine','Revenue (Millions USD)'] = 1000
team1.at['Harry Potter and the sorcers stone','Revenue (Millions USD)'] = 1000
team1.at['Harry Potter and the deathly hallows I','Revenue (Millions USD)'] = 1300
team1['Revenue (Millions USD)'] = team1['Revenue (Millions USD)'].astype(float)
team1['Revenue (Millions USD)'].unique()

array([2.08440e+02, 5.66900e+01, 2.14940e+02, 9.64500e+01, 2.62820e+02,
       9.62540e+02, 5.22000e+00, 6.95220e+02, 6.78220e+02,         nan,
       2.99160e+02, 2.12740e+02, 2.30880e+02, 1.71260e+02, 4.88620e+02,
       8.43800e+01, 3.46000e+02, 1.67100e+03, 1.15000e+02, 4.90000e+01,
       3.00000e+02, 9.75700e+01, 2.13000e+02, 2.05000e+02, 4.57000e+02,
       3.92000e+02, 1.23000e+02, 2.26940e+02, 1.95000e+02, 2.63000e+02,
       8.21000e+02, 8.75000e+02, 1.02300e+03, 2.07000e+02, 5.00000e+01,
       3.52000e+02, 7.09000e+02, 1.17000e+02, 1.00000e+02, 1.13100e+02,
       1.30160e+02, 2.71450e+02, 3.20410e+02, 2.57580e+02, 4.08490e+02,
       1.37495e+03, 1.00500e+02, 6.69000e+00, 4.63400e+02, 6.25400e+01,
       2.92370e+03, 5.92470e+02, 5.69620e+02, 7.59850e+02, 8.59100e+02,
       2.33500e+02, 1.41780e+02, 5.85800e+02, 6.23930e+02, 4.49320e+02,
       1.65330e+02, 1.02546e+03, 1.05430e+03, 1.27000e+00, 4.74960e+02,
       2.77943e+03, 5.43280e+02, 3.94430e+02, 3.66080e+02, 5.797

In [17]:
team1['P9'].unique()

array(['90', 'NA ', '100', '77', '89', '73', '85', '68', '93', '76', '99',
       '80', '74', '87', '96', '72', '98', '88', '92', '94', '79', '66',
       '97', '81', '91', '84', '86', '95', '83', '60', '75', '70', '65',
       '20', '67', '50', nan, '69', '82', '29', '78', '58', '56', '71'],
      dtype=object)

There is an entry with a trailing space, `'NA '`, that we can subsitute for `nan`

In [18]:
team1['P9']=team1['P9'].replace('NA ', nan)
team1['P9']=team1['P9'].astype(float)
team1['P9'].unique()

array([ 90.,  nan, 100.,  77.,  89.,  73.,  85.,  68.,  93.,  76.,  99.,
        80.,  74.,  87.,  96.,  72.,  98.,  88.,  92.,  94.,  79.,  66.,
        97.,  81.,  91.,  84.,  86.,  95.,  83.,  60.,  75.,  70.,  65.,
        20.,  67.,  50.,  69.,  82.,  29.,  78.,  58.,  56.,  71.])

Now, let's explore the `'Genre'` column to see if there are invisible characters.

In [19]:
team1['Genre'].unique()

array(['Romance ', 'Comedy', 'Romance', 'Adventure', 'Action', 'Drama ',
       'Thriller', 'Fiction ', 'Drama', 'Biography', 'Thriller ',
       'Action ', 'Fantasy ', 'Animation', 'Horror', 'Fantacy',
       'Adventure ', 'Fantasy', 'Trhiller', 'Crime', 'Mystery',
       'Sience Fiction', 'Crime ', 'Science fiction', 'Suspense',
       'Romantic', 'Musical', 'Musical '], dtype=object)

There are trailing white spaces on some entries, as well as a few spelling mistakes. Let's clean this column.

In [20]:
team1['Genre'] = team1['Genre'].str.replace(' ','')
team1['Genre'] = team1['Genre'].replace('Fantacy','Fantasy')
team1['Genre'] = team1['Genre'].replace('Trhiller','Thriller')
team1['Genre'] = team1['Genre'].replace('SienceFiction','Sci-Fi')
team1['Genre'] = team1['Genre'].replace('ScienceFiction','Sci-Fi')
team1['Genre'] = team1['Genre'].replace('Sciencefiction','Sci-Fi')
team1['Genre'].unique()

array(['Romance', 'Comedy', 'Adventure', 'Action', 'Drama', 'Thriller',
       'Fiction', 'Biography', 'Fantasy', 'Animation', 'Horror', 'Crime',
       'Mystery', 'Sci-Fi', 'Suspense', 'Romantic', 'Musical'],
      dtype=object)

Our data is clean, with the correct data types.

In [21]:
team1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 201 entries, Me Before you  to Men in black 2
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Year                    201 non-null    int64  
 1   Runtime (Minutes)       201 non-null    int64  
 2   Genre                   201 non-null    object 
 3   Budget (Millions USD)   199 non-null    float64
 4   Revenue (Millions USD)  197 non-null    float64
 5   IMDb                    201 non-null    int64  
 6   Metacritic              200 non-null    float64
 7   Rotten Tomatoes         200 non-null    float64
 8   Filmaffinity            200 non-null    float64
 9   Personal                198 non-null    float64
 10  Average                 201 non-null    float64
 11  P1                      166 non-null    float64
 12  P2                      182 non-null    float64
 13  P3                      166 non-null    float64
 14  P4                     

Now, let's verify that the rating columns have the correct scale

In [22]:
team1.iloc[:,5:10].describe()

,IMDb,Metacritic,Rotten Tomatoes,Filmaffinity,Personal
count,201.000000,200.000000,200.000000,200.000000,198.000000
mean,72.796020,72.575000,77.170000,64.675000,82.116162
std,9.934444,13.143368,15.604763,11.261147,13.716552
min,40.000000,30.000000,27.000000,35.000000,30.000000
25%,66.000000,65.000000,70.750000,57.000000,73.250000
50%,73.000000,75.000000,81.000000,65.000000,85.000000
75%,80.000000,83.000000,89.000000,74.000000,91.000000
max,97.000000,96.000000,98.000000,90.000000,100.000000


In [23]:
team1.iloc[:,10:20].describe()

,Average,P1,P2,P3,P4,P5,P6,P7,P8,P9
count,201.000000,166.000000,182.000000,166.000000,200.000000,173.000000,183.000000,185.000000,186.000000,188.000000
mean,83.266169,84.614458,82.390110,81.415663,85.480000,84.982659,83.404372,78.189189,77.731183,82.595745
std,5.087293,11.230188,14.627474,14.576926,9.020891,12.101064,11.055822,17.491982,18.710390,13.033744
min,71.000000,45.000000,24.000000,28.000000,25.000000,40.000000,26.000000,22.000000,13.000000,20.000000
25%,79.500000,78.000000,75.000000,72.500000,79.750000,78.000000,77.500000,70.000000,72.000000,75.000000
50%,83.900000,87.000000,86.500000,84.500000,85.000000,87.000000,85.000000,82.000000,82.000000,86.000000
75%,86.700000,92.750000,92.000000,90.750000,92.000000,95.000000,92.000000,91.000000,90.000000,91.000000
max,94.400000,100.000000,100.000000,100.000000,99.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [24]:
team1.iloc[:,20:].describe()

,P10,P11,P12,P13,P14,P15,P16,P17,P18,P19,P20
count,186.000000,184.000000,201.000000,185.000000,193.000000,193.000000,197.000000,195.000000,194.000000,190.000000,188.000000
mean,85.107527,82.858696,85.815920,85.859459,85.373057,79.181347,80.730964,84.548718,84.716495,84.678947,84.739362
std,10.569167,11.906816,7.991304,9.786955,9.956975,17.024286,16.331505,10.013982,10.910709,9.910557,11.666999
min,47.000000,45.000000,67.000000,46.000000,44.000000,9.000000,12.000000,43.000000,25.000000,30.000000,34.000000
25%,79.000000,76.000000,79.000000,78.000000,78.000000,73.000000,76.000000,78.000000,80.000000,79.000000,78.000000
50%,88.000000,85.000000,86.000000,87.000000,88.000000,84.000000,85.000000,88.000000,85.000000,87.000000,87.500000
75%,92.000000,91.000000,92.000000,94.000000,92.000000,90.000000,91.000000,91.500000,92.000000,91.000000,93.000000
max,100.000000,100.000000,99.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [25]:
team1.to_csv('../datasets/team1_clean.csv')

### Team 2

The first order of business is loading the dataset into a `pandas DataFrame` object.

In [26]:
team2 = pd.read_csv('../datasets/team2.csv')

Let’s explore our database so we can get a glimpse at the data

In [27]:
team2.head()

,Title,Year,Duration(min),Genre,Budget(Miill.),Revenue(Mill.),Imdb,Critic_1,Critic_2,Critic_3,...,Fans 11,Fans 12,Fans 13,Fans 14,Fans 15,Fans 16,Fans 17,Fans 18,Fans 19,Fans 20
0,The outsiders,1983,91,Crime,10,25.8,70,80,70.0,60,...,NaN,80.0,NaN,na,80.0,NaN,NaN,70.0,90.0,73.0
1,Little women,2019,135,Drama,40,218.8,78,60,90.0,80,...,100.0,100.0,100.0,na,75.0,90.0,NaN,85.0,90.0,79.0
2,Pride and prejudice,2005,129,Drama,28,121.6,78,80,90.0,80,...,70.0,100.0,100.0,na,NaN,85.0,NaN,86.0,100.0,84.0
3,The notebook,2004,123,Romance,29,118.2,78,80,80.0,70,...,100.0,100.0,100.0,95,70.0,90.0,NaN,87.0,100.0,88.0
4,Coraline,2009,100,Fantasy,60,131.7,77,70,80.0,100,...,100.0,100.0,100.0,100,95.0,100.0,100.0,85.0,90.0,90.0


We can check the dimensions of our dataframe with `pd.shape`.

In [28]:
team2.shape

(200, 32)

We observe that there are 200 observations with 32 attributes.

#### Column cleaning

In this section we will clean the column names so they look homogeneous, without extra (invisible) characters.

First of all, we print a list of column names.

In [29]:
team2.columns

Index(['Title', 'Year', 'Duration(min)', 'Genre', 'Budget(Miill.)',
       'Revenue(Mill.)', 'Imdb', 'Critic_1', 'Critic_2', 'Critic_3',
       'Personal', 'Fan average', 'Fans 1', 'Fans 2', 'Fans 3', 'Fans 4',
       'Fans 5', 'Fans 6', 'Fans 7', 'Fans 8', 'Fans 9', 'Fans 10', 'Fans 11',
       'Fans 12', 'Fans 13', 'Fans 14', 'Fans 15', 'Fans 16', 'Fans 17',
       'Fans 18', 'Fans 19', 'Fans 20'],
      dtype='object')

We will capitalize all column names and correct some minor mistakes.

In [30]:
team2.rename(columns = {'Duration(min)':'Runtime (Min)', 'Budget(Miill.)':'Budget (Mill)', 'Revenue(Mill.)':'Revenue (Mill)', 'Imdb':'IMDb'}, inplace=True)
team2.columns

Index(['Title', 'Year', 'Runtime (Min)', 'Genre', 'Budget (Mill)',
       'Revenue (Mill)', 'IMDb', 'Critic_1', 'Critic_2', 'Critic_3',
       'Personal', 'Fan average', 'Fans 1', 'Fans 2', 'Fans 3', 'Fans 4',
       'Fans 5', 'Fans 6', 'Fans 7', 'Fans 8', 'Fans 9', 'Fans 10', 'Fans 11',
       'Fans 12', 'Fans 13', 'Fans 14', 'Fans 15', 'Fans 16', 'Fans 17',
       'Fans 18', 'Fans 19', 'Fans 20'],
      dtype='object')

Finally, we will set the `Title` column as the indexing column of our dataframe.

In [31]:
team2.set_index('Title', inplace=True)

#### Data cleaning

We will follow the guidelines stated below for cleaning the data for each column:
- `Year`: This column will be left unchanged. Data type can be `str` or `int`.
- `Runtime`: Data type should be `int`. Remove any indication to a unit of measurement.
- `Genre`: Data type should be `str`. We will capitalize the data, and if there is more than one genre, we will keep the first one and remove the rest.
- `Budget`: Data type can be `int` or `float`. To that end, we will remove dollar signs as well as the thousands separator. We will scale the data so the unit of measurment is millions of dollars.
- `Revenue`: Same treatment as that of `Budget`.
- Rating columns: Data type can be `int` or `float`. Change scale to [0,100].

Let's explore the type of each of the columns. In the table below we can also see if there is missing data.

In [32]:
team2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, The outsiders to Anyone But You
Data columns (total 31 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Year            200 non-null    int64  
 1   Runtime (Min)   200 non-null    int64  
 2   Genre           200 non-null    object 
 3   Budget (Mill)   200 non-null    object 
 4   Revenue (Mill)  199 non-null    object 
 5   IMDb            200 non-null    int64  
 6   Critic_1        200 non-null    int64  
 7   Critic_2        199 non-null    float64
 8   Critic_3        200 non-null    int64  
 9   Personal        159 non-null    float64
 10  Fan average     200 non-null    float64
 11  Fans 1          185 non-null    float64
 12  Fans 2          174 non-null    float64
 13  Fans 3          163 non-null    float64
 14  Fans 4          170 non-null    float64
 15  Fans 5          154 non-null    float64
 16  Fans 6          162 non-null    float64
 17  Fans 7          1

We observe that columns `Budget`, `Revenue`, and `Fans 14` have the wrong type. Let's explore each column in turn, and convert them to `int` or `float` as appropriate.

In [33]:
team2['Budget (Mill)'].unique()

array(['10', '40', '28', '29', '60', '90', '35', '50', '19', '150', '14',
       '7.5', '11.4', '20', '4.5', '200', '100', '14.3', '17', '78',
       '190', '170', '165', '55', '30', '63', '46', '185', '92', '175',
       '25', '65', '3.7', '130', '80', '24', '5.1', '294.7', '160', '34',
       '19.4', '3.3', '18', '11', '32', '2', '17.1', '97', '58', '225',
       '33', '9.5', '95', '260', '105', '16.4', ' $6.00 ', ' $0.95 ',
       ' $103.00 ', ' $72.00 ', ' $45.00 ', ' $63.00 ', ' $10.50 ',
       ' $11.00 ', ' $18.00 ', ' $42.00 ', ' $125.00 ', ' $100.00 ',
       ' $130.00 ', ' $150.00 ', ' $250.00 ', ' $93.00 ', ' $94.00 ',
       ' $220.00 ', ' $325.00 ', ' $356.00 ', ' $140.00 ', ' $170.00 ',
       ' $365.00 ', ' $200.00 ', ' $165.00 ', ' $180.00 ', ' $60.00 ',
       ' $75.00 ', ' $110.00 ', ' $40.00 ', ' $135.00 ', ' $20.00 ',
       ' $25.00 ', ' $1.50 ', ' $52.00 ', ' $35.00 ', ' $33.00 ',
       ' $90.00 ', ' $19.00 ', ' $28.00 ', ' $2.00 ', ' $13.00 ',
       ' $41.00 ',

For the `Budget` column we need to remove the dollar sign and the extra spaces, and then convert the column to `float`.

In [34]:
team2['Budget (Mill)'] = team2['Budget (Mill)'].str.replace('$','')
team2['Budget (Mill)'] = team2['Budget (Mill)'].str.replace(' ','')
team2['Budget (Mill)'] = team2['Budget (Mill)'].astype(float)
team2['Budget (Mill)'].unique()

array([ 10.  ,  40.  ,  28.  ,  29.  ,  60.  ,  90.  ,  35.  ,  50.  ,
        19.  , 150.  ,  14.  ,   7.5 ,  11.4 ,  20.  ,   4.5 , 200.  ,
       100.  ,  14.3 ,  17.  ,  78.  , 190.  , 170.  , 165.  ,  55.  ,
        30.  ,  63.  ,  46.  , 185.  ,  92.  , 175.  ,  25.  ,  65.  ,
         3.7 , 130.  ,  80.  ,  24.  ,   5.1 , 294.7 , 160.  ,  34.  ,
        19.4 ,   3.3 ,  18.  ,  11.  ,  32.  ,   2.  ,  17.1 ,  97.  ,
        58.  , 225.  ,  33.  ,   9.5 ,  95.  , 260.  , 105.  ,  16.4 ,
         6.  ,   0.95, 103.  ,  72.  ,  45.  ,  10.5 ,  42.  , 125.  ,
       250.  ,  93.  ,  94.  , 220.  , 325.  , 356.  , 140.  , 365.  ,
       180.  ,  75.  , 110.  , 135.  ,   1.5 ,  52.  ,  13.  ,  41.  ,
       237.  , 460.  , 120.  ,  85.  ,  12.  ,  74.  ,  65.5 ,  37.  ,
        36.  ,  70.  ,  84.  ,   4.2 ,   7.  ])

In [35]:
team2['Revenue (Mill)'].unique()

array(['25.8', '218.8', '121.6', '118.2', '131.7', '304.3', '107.1',
       '469.3', '177.5', '272.7', '71.2', '384.2', '690.8', '349.5',
       '60.8', '262.1', '256', '255', '2,264.70', '975.4', '143.4',
       '115.6', '695.2', '711.8', '773.3', '35', '407', '358.1', '87.8',
       '705.2', '1,078.90', '678.2', '471.9', '467.8', '47.5', '122.1',
       '1,495.70', '722.2', '631.7', '859.1', '172.4', '84.8', '31.1',
       '632.1', '294.9', '101.3', '238', '21.7', '383.9', '439.4',
       '497.3', '1,067.30', '1,073.80', '1,445.60', '488.6', '932.3',
       '813.3', '752.6', '631.6', '1,243.20', '494.8', '226.9', '348.3',
       '839', '195.2', '49.4', '859', '103.2', '18.1', '340.9', '50',
       '93.6', '15.8', '28.2', '133.4', '773', '10.9', '187.7', nan,
       '312.8', '619.1', '782.8', '670.1', '772.2', '3.2', '264.1',
       '83.8', '151.6', '327.3', '426.6', '316.8', '542.4', '592.5',
       '267.1', '291.5', '235.8', ' $250.00 ', ' $4.00 ', ' $457.60 ',
       ' $210.40 ', '

For the `Revenue` column we need to remove the dollar sign, the thousands separator, and the extra spaces, and then convert the column to `float`.

In [36]:
team2['Revenue (Mill)'] = team2['Revenue (Mill)'].str.replace('$','')
team2['Revenue (Mill)'] = team2['Revenue (Mill)'].str.replace(' ','')
team2['Revenue (Mill)'] = team2['Revenue (Mill)'].str.replace(',','')
team2['Revenue (Mill)'] = team2['Revenue (Mill)'].astype(float)
team2['Revenue (Mill)'].unique()

array([  25.8 ,  218.8 ,  121.6 ,  118.2 ,  131.7 ,  304.3 ,  107.1 ,
        469.3 ,  177.5 ,  272.7 ,   71.2 ,  384.2 ,  690.8 ,  349.5 ,
         60.8 ,  262.1 ,  256.  ,  255.  , 2264.7 ,  975.4 ,  143.4 ,
        115.6 ,  695.2 ,  711.8 ,  773.3 ,   35.  ,  407.  ,  358.1 ,
         87.8 ,  705.2 , 1078.9 ,  678.2 ,  471.9 ,  467.8 ,   47.5 ,
        122.1 , 1495.7 ,  722.2 ,  631.7 ,  859.1 ,  172.4 ,   84.8 ,
         31.1 ,  632.1 ,  294.9 ,  101.3 ,  238.  ,   21.7 ,  383.9 ,
        439.4 ,  497.3 , 1067.3 , 1073.8 , 1445.6 ,  488.6 ,  932.3 ,
        813.3 ,  752.6 ,  631.6 , 1243.2 ,  494.8 ,  226.9 ,  348.3 ,
        839.  ,  195.2 ,   49.4 ,  859.  ,  103.2 ,   18.1 ,  340.9 ,
         50.  ,   93.6 ,   15.8 ,   28.2 ,  133.4 ,  773.  ,   10.9 ,
        187.7 ,     nan,  312.8 ,  619.1 ,  782.8 ,  670.1 ,  772.2 ,
          3.2 ,  264.1 ,   83.8 ,  151.6 ,  327.3 ,  426.6 ,  316.8 ,
        542.4 ,  592.5 ,  267.1 ,  291.5 ,  235.8 ,  250.  ,    4.  ,
        457.6 ,  210

In [37]:
team2['Fans 14'].unique()

array(['na', '95', '100', '80', '70', '90', '85', 'na80', '92', nan, '81',
       '94', '77', '82', '55', '62', '88', '78', '63', '84', '75', '72',
       '87', '57', '76', '66', '83', '65', '68', '69', '91', '59', '86',
       '96', '97', '20', '50', '29'], dtype=object)

In [38]:
print(team2[team2['Fans 14']=='na80'].index.values)

['The Hunger Games']


For the `Fans 14` column we need to replace `'na'` with `nan`, `'na80'` with `80`, and convert it to `float`.

In [39]:
team2['Fans 14']=team2['Fans 14'].replace('na80', 80)
team2['Fans 14']=team2['Fans 14'].replace('na', nan)
team2['Fans 14']=team2['Fans 14'].astype(float)
team2['Fans 14'].unique()

array([ nan,  95., 100.,  80.,  70.,  90.,  85.,  92.,  81.,  94.,  77.,
        82.,  55.,  62.,  88.,  78.,  63.,  84.,  75.,  72.,  87.,  57.,
        76.,  66.,  83.,  65.,  68.,  69.,  91.,  59.,  86.,  96.,  97.,
        20.,  50.,  29.])

Now, let's explore the `'Genre'` column to see if there are invisible characters.

In [40]:
team2['Genre'].unique()

array(['Crime', 'Drama', 'Romance', 'Fantasy', 'Adventure', 'Action',
       'Comedy', 'Thriller', 'Horror', 'Sci-Fi', 'Coming-of-age',
       'Family', 'Satire', 'Docudrama', 'Caper', 'Dark Fantasy',
       'Superhero', 'Romantic Comedy', 'Cop Drama', 'Fairy Tale',
       'Historical', 'Animation', 'Suspense', 'Comdey', 'Science Fiction',
       'Musical'], dtype=object)

`Science Fiction` and `Sci-Fi` are the same category. Also, `Comdey` should be `Comedy`.

In [41]:
team2['Genre'] = team2['Genre'].replace('Science Fiction','Sci-Fi')
team2['Genre'] = team2['Genre'].replace('Comdey','Comedy')
team2['Genre'].unique()

array(['Crime', 'Drama', 'Romance', 'Fantasy', 'Adventure', 'Action',
       'Comedy', 'Thriller', 'Horror', 'Sci-Fi', 'Coming-of-age',
       'Family', 'Satire', 'Docudrama', 'Caper', 'Dark Fantasy',
       'Superhero', 'Romantic Comedy', 'Cop Drama', 'Fairy Tale',
       'Historical', 'Animation', 'Suspense', 'Musical'], dtype=object)

Our data is clean, with the correct data types.

In [42]:
team2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, The outsiders to Anyone But You
Data columns (total 31 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Year            200 non-null    int64  
 1   Runtime (Min)   200 non-null    int64  
 2   Genre           200 non-null    object 
 3   Budget (Mill)   200 non-null    float64
 4   Revenue (Mill)  199 non-null    float64
 5   IMDb            200 non-null    int64  
 6   Critic_1        200 non-null    int64  
 7   Critic_2        199 non-null    float64
 8   Critic_3        200 non-null    int64  
 9   Personal        159 non-null    float64
 10  Fan average     200 non-null    float64
 11  Fans 1          185 non-null    float64
 12  Fans 2          174 non-null    float64
 13  Fans 3          163 non-null    float64
 14  Fans 4          170 non-null    float64
 15  Fans 5          154 non-null    float64
 16  Fans 6          162 non-null    float64
 17  Fans 7          1

Now, let's verify that the rating columns have the correct scale

In [43]:
team2.iloc[:,5:10].describe()

,IMDb,Critic_1,Critic_2,Critic_3,Personal
count,200.000000,200.00000,199.000000,200.000000,159.000000
mean,76.240000,81.21500,79.251256,79.620000,84.106918
std,9.666936,14.22217,14.417904,13.078257,11.300045
min,44.000000,10.00000,10.000000,10.000000,50.000000
25%,72.000000,75.00000,72.000000,74.750000,80.000000
50%,78.000000,84.00000,80.000000,80.000000,85.000000
75%,82.000000,90.25000,90.000000,89.250000,90.000000
max,95.000000,100.00000,100.000000,100.000000,100.000000


In [44]:
team2.iloc[:,10:20].describe()

,Fan average,Fans 1,Fans 2,Fans 3,Fans 4,Fans 5,Fans 6,Fans 7,Fans 8,Fans 9
count,200.000000,185.000000,174.000000,163.000000,170.000000,154.000000,162.000000,151.000000,171.000000,172.000000
mean,82.760817,83.724324,81.344828,79.926380,81.664706,82.922078,82.740741,81.052980,83.105263,83.145349
std,7.451353,19.974296,19.622346,22.673081,17.496219,16.954970,14.596245,16.390562,14.696559,14.113029
min,38.800000,2.000000,10.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,79.025000,80.000000,78.000000,77.500000,77.000000,79.250000,79.000000,77.000000,80.000000,80.000000
50%,82.400000,90.000000,88.000000,89.000000,85.000000,87.000000,84.500000,83.000000,82.000000,85.000000
75%,87.947222,100.000000,95.000000,95.000000,93.000000,92.000000,90.750000,90.500000,92.500000,92.000000
max,97.687500,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [45]:
team2.iloc[:,20:].describe()

,Fans 10,Fans 11,Fans 12,Fans 13,Fans 14,Fans 15,Fans 16,Fans 17,Fans 18,Fans 19,Fans 20
count,160.000000,170.000000,184.000000,158.000000,152.000000,164.000000,166.000000,156.000000,181.000000,181.000000,182.000000
mean,82.562500,84.217647,81.559783,84.620253,83.144737,83.463415,82.813253,85.237179,84.486188,85.524862,83.368132
std,12.992348,14.342998,16.217332,12.797595,13.536513,11.990482,12.285298,12.740138,12.869692,13.050998,12.120558
min,1.000000,20.000000,0.000000,20.000000,20.000000,20.000000,20.000000,20.000000,30.000000,30.000000,30.000000
25%,76.000000,80.000000,77.000000,79.000000,80.000000,79.750000,79.000000,80.000000,78.000000,79.000000,77.000000
50%,83.000000,85.000000,81.000000,85.000000,82.000000,84.000000,83.000000,85.000000,86.000000,87.000000,83.000000
75%,90.000000,95.000000,90.000000,94.000000,91.250000,91.000000,91.000000,95.000000,94.000000,95.000000,92.000000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [46]:
team2.to_csv('../datasets/team2_clean.csv')

### Team 3

The first order of business is loading the dataset into a `pandas DataFrame` object.

In [47]:
team3 = pd.read_csv('../datasets/team3.csv')

Let’s explore our database so we can get a glimpse at the data

In [48]:
team3.head()

,Name,Year,Duration (min),Genre,Budget (Million USD),USA Revenue (Million USD),Global Revenue (Million USD),IMDb,Rotten tomatoes,Metacritic,...,Fan 20,Fan 21,Fan 22,Fan 23,Fan 24,Fan 25,Fan 26,Fan 27,Fan 28,Fan 29
0,Pulp Fiction (1994),1994,154,Crime,8.0,107.93,213.93,89,92,95,...,NaN,NaN,95,90.0,90.0,NaN,97,80.0,100,100.0
1,The Shawshank Redemption (1994),1994,142,Thriller,25.0,28.76,29.33,93,89,82,...,NaN,NaN,90,90.0,78.0,NaN,81,70.0,0,60.0
2,Forrest Gump (1994),1994,142,Comedy,55.0,330.46,678.22,88,76,82,...,100.0,100.0,90,50.0,83.0,NaN,84,80.0,100,100.0
3,The Lion King (1994),1994,88,Family,45.0,424.98,979.05,85,92,88,...,100.0,100.0,85,50.0,63.0,100.0,92,60.0,100,100.0
4,Titanic (1997),1997,194,Romance,200.0,674.29,2264.75,79,88,75,...,100.0,100.0,85,50.0,55.0,NaN,84,65.0,70,100.0


We can check the dimensions of our dataframe with `pd.shape`.

In [49]:
team3.shape

(180, 42)

We observe that there are 180 observations with 42 attributes.

#### Column cleaning

In this section we will clean the column names so they look homogeneous, without extra (invisible) characters.

First of all, we print a list of column names.

In [50]:
team3.columns

Index(['Name', 'Year', 'Duration (min)', 'Genre', 'Budget (Million USD)',
       'USA Revenue (Million USD)', 'Global Revenue (Million USD)', 'IMDb',
       'Rotten tomatoes', 'Metacritic', 'Filmaffinity', 'Personal',
       'Fans Average', 'Fan 1', 'Fan 2', 'Fan 3', 'Fan 4', 'Fan 5', 'Fan 6',
       'Fan 7', 'Fan 8', 'Fan 9', 'Fan 10', 'Fan 11', 'Fan 12', 'Fan 13',
       'Fan 14', 'Fan 15', 'Fan 16', 'Fan 17', 'Fan 18', 'Fan 19', 'Fan 20',
       'Fan 21', 'Fan 22', 'Fan 23', 'Fan 24', 'Fan 25', 'Fan 26', 'Fan 27',
       'Fan 28', 'Fan 29'],
      dtype='object')

We will capitalize all column names and correct some spelling mistakes.

In [51]:
team3.rename(columns = {'Name':'Title', 'Duration (min)':'Runtime (min)', 'Rotten tomatoes':'Rotten Tomatoes'}, inplace=True)
team3.columns

Index(['Title', 'Year', 'Runtime (min)', 'Genre', 'Budget (Million USD)',
       'USA Revenue (Million USD)', 'Global Revenue (Million USD)', 'IMDb',
       'Rotten Tomatoes', 'Metacritic', 'Filmaffinity', 'Personal',
       'Fans Average', 'Fan 1', 'Fan 2', 'Fan 3', 'Fan 4', 'Fan 5', 'Fan 6',
       'Fan 7', 'Fan 8', 'Fan 9', 'Fan 10', 'Fan 11', 'Fan 12', 'Fan 13',
       'Fan 14', 'Fan 15', 'Fan 16', 'Fan 17', 'Fan 18', 'Fan 19', 'Fan 20',
       'Fan 21', 'Fan 22', 'Fan 23', 'Fan 24', 'Fan 25', 'Fan 26', 'Fan 27',
       'Fan 28', 'Fan 29'],
      dtype='object')

Finally, we will set the `Title` column as the indexing column of our dataframe.

In [52]:
team3.set_index('Title', inplace=True)

#### Data cleaning

We will follow the guidelines stated below for cleaning the data for each column:
- `Year`: This column will be left unchanged. Data type can be `str` or `int`.
- `Runtime`: Data type should be `int`. Remove any indication to a unit of measurement.
- `Genre`: Data type should be `str`. We will capitalize the data, and if there is more than one genre, we will keep the first one and remove the rest.
- `Budget`: Data type can be `int` or `float`. To that end, we will remove dollar signs as well as the thousands separator. We will scale the data so the unit of measurment is millions of dollars.
- `Revenue`: Same treatment as that of `Budget`.
- Rating columns: Data type can be `int` or `float`. Change scale to [0,100].

Let's explore the type of each of the columns. In the table below we can also see if there is missing data.

In [53]:
team3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 180 entries, Pulp Fiction (1994)   to Star Wars: The Force Awakens (2015)
Data columns (total 41 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Year                          180 non-null    int64  
 1   Runtime (min)                 180 non-null    int64  
 2   Genre                         180 non-null    object 
 3   Budget (Million USD)          180 non-null    float64
 4   USA Revenue (Million USD)     179 non-null    float64
 5   Global Revenue (Million USD)  180 non-null    object 
 6   IMDb                          180 non-null    int64  
 7   Rotten Tomatoes               180 non-null    int64  
 8   Metacritic                    180 non-null    int64  
 9   Filmaffinity                  179 non-null    float64
 10  Personal                      161 non-null    float64
 11  Fans Average                  180 non-null    float64
 12  Fan 1            

We observe that columns `Global Revenue`, and `Fan 3, 4, 5, 7, 8, 10, 13, 15, 22, 26` have the wrong type. Let's explore each column in turn, and convert them to `int` or `float` as appropriate.

In [54]:
team3['Global Revenue (Million USD)'].unique()

array(['213.93', '29.33', '678.22', '979.05', '2264.75', '482.35',
       '467.84', '465.39', '40.05', '887.43', '937.29', '1151.46',
       '941.64', '654.26', '2923.71', '1008.52', '291.48', '171.63',
       '378.41', '839.03', '224.93', '394.44', '1342.48', '1520.54',
       '1306', '773.03', '380.44', '471.99', '255.75', '1374.96',
       '2799.44', '262.13', '1078.96', '1922.6', '407.58', '143.41',
       '1495.7', '975.3', '1445.64', '86', '773.35', '705.2', '535.95',
       '782.84', '823.97', '2052.42', '367.8', '340.96', '390.411',
       '690.89', '407.04', '195.24', '276.61', '312.89', '814.64', '8',
       '93.69', '46.33', '772.25', '1132.68', '462', '379.75', '130.42',
       '121.98', '141.94', '233.5', '470.12', '962.54', '192.91',
       '774.15', '157.39', '955.76', '365', '230.5', '331.5', '204.3',
       '800.1', '169.6', '726', '436.2', '1,152', '348.3', '677.9',
       '288.9', '321.3', '755.4', '661.4', '1,051', '69.9', '432.2',
       '1,159', '1,035', '634', '8

For the `Global Revenue` column we need to remove the thousands separator and then convert the column to `float`.

In [55]:
team3['Global Revenue (Million USD)'] = team3['Global Revenue (Million USD)'].str.replace(',','')
team3['Global Revenue (Million USD)'] = team3['Global Revenue (Million USD)'].astype(float)
team3['Global Revenue (Million USD)'].unique()

array([2.13930e+02, 2.93300e+01, 6.78220e+02, 9.79050e+02, 2.26475e+03,
       4.82350e+02, 4.67840e+02, 4.65390e+02, 4.00500e+01, 8.87430e+02,
       9.37290e+02, 1.15146e+03, 9.41640e+02, 6.54260e+02, 2.92371e+03,
       1.00852e+03, 2.91480e+02, 1.71630e+02, 3.78410e+02, 8.39030e+02,
       2.24930e+02, 3.94440e+02, 1.34248e+03, 1.52054e+03, 1.30600e+03,
       7.73030e+02, 3.80440e+02, 4.71990e+02, 2.55750e+02, 1.37496e+03,
       2.79944e+03, 2.62130e+02, 1.07896e+03, 1.92260e+03, 4.07580e+02,
       1.43410e+02, 1.49570e+03, 9.75300e+02, 1.44564e+03, 8.60000e+01,
       7.73350e+02, 7.05200e+02, 5.35950e+02, 7.82840e+02, 8.23970e+02,
       2.05242e+03, 3.67800e+02, 3.40960e+02, 3.90411e+02, 6.90890e+02,
       4.07040e+02, 1.95240e+02, 2.76610e+02, 3.12890e+02, 8.14640e+02,
       8.00000e+00, 9.36900e+01, 4.63300e+01, 7.72250e+02, 1.13268e+03,
       4.62000e+02, 3.79750e+02, 1.30420e+02, 1.21980e+02, 1.41940e+02,
       2.33500e+02, 4.70120e+02, 9.62540e+02, 1.92910e+02, 7.741

In [56]:
team3['Fan 3'].unique()

array([nan, '90', '85', '70', '95', '80', '100', '60', '9', 'Na'],
      dtype=object)

For the `Fan 3` column we need to change an `Na` to `nan` and then convert the column to `float`.

In [57]:
team3['Fan 3'] = team3['Fan 3'].replace('Na',nan)
team3['Fan 3'] = team3['Fan 3'].astype(float)
team3['Fan 3'].unique()

array([ nan,  90.,  85.,  70.,  95.,  80., 100.,  60.,   9.])

We proceed in a similar manner for the remaining Fan columns, without explaining each step.

In [58]:
team3['Fan 4'].unique()

array([nan, 'Na', '70', '100', '98', '95', '85', '90', '97', '96', '80',
       '10', '79'], dtype=object)

In [59]:
team3['Fan 4'] = team3['Fan 4'].replace('Na',nan)
team3['Fan 4'] = team3['Fan 4'].astype(float)
team3['Fan 4'].unique()

array([ nan,  70., 100.,  98.,  95.,  85.,  90.,  97.,  96.,  80.,  10.,
        79.])

In [60]:
team3['Fan 5'].unique()

array([nan, 'NA ', '100', '90', '95', '70', '80', '85', '50', '0'],
      dtype=object)

In [61]:
team3['Fan 5'] = team3['Fan 5'].replace('NA ',nan)
team3['Fan 5'] = team3['Fan 5'].astype(float)
team3['Fan 5'].unique()

array([ nan, 100.,  90.,  95.,  70.,  80.,  85.,  50.,   0.])

In [62]:
team3['Fan 7'].unique()

array(['na', '80', '70', '90', '100', nan], dtype=object)

In [63]:
team3['Fan 7'] = team3['Fan 7'].replace('na',nan)
team3['Fan 7'] = team3['Fan 7'].astype(float)
team3['Fan 7'].unique()

array([ nan,  80.,  70.,  90., 100.])

In [64]:
team3['Fan 8'].unique()

array(['90', 'Na', '100', '80', '95', ' Na', '85', '75', '50', '60', '70',
       nan], dtype=object)

In [65]:
team3['Fan 8'] = team3['Fan 8'].replace('Na',nan)
team3['Fan 8'] = team3['Fan 8'].replace(' Na',nan)
team3['Fan 8'] = team3['Fan 8'].astype(float)
team3['Fan 8'].unique()

array([ 90.,  nan, 100.,  80.,  95.,  85.,  75.,  50.,  60.,  70.])

In [66]:
team3['Fan 10'].unique()

array([nan, '87', '100', '90', '95', '97', '96', '85', 'na', '86', '98'],
      dtype=object)

In [67]:
team3['Fan 10'] = team3['Fan 10'].replace('na',nan)
team3['Fan 10'] = team3['Fan 10'].astype(float)
team3['Fan 10'].unique()

array([ nan,  87., 100.,  90.,  95.,  97.,  96.,  85.,  86.,  98.])

In [68]:
team3['Fan 13'].unique()

array(['97', '94', '87', nan, '86', '90', '98', '80', '91', '93', '88',
       '79', '100', '95', '96', '89', '92', '81', '84', '73', '85', '82',
       'Na', '83', '70', '74', '78', '77', '76'], dtype=object)

In [69]:
team3['Fan 13'] = team3['Fan 13'].replace('Na',nan)
team3['Fan 13'] = team3['Fan 13'].astype(float)
team3['Fan 13'].unique()

array([ 97.,  94.,  87.,  nan,  86.,  90.,  98.,  80.,  91.,  93.,  88.,
        79., 100.,  95.,  96.,  89.,  92.,  81.,  84.,  73.,  85.,  82.,
        83.,  70.,  74.,  78.,  77.,  76.])

In [70]:
team3['Fan 15'].unique()

array(['90', nan, '80', '70', '100', 'Na', '85', '75', 'nA', '78', '50',
       '67', '57', '59', '47', '60', '79', '49', '40', '68', '87', '45'],
      dtype=object)

In [71]:
team3['Fan 15'] = team3['Fan 15'].replace('NA ',nan)
team3['Fan 15'] = team3['Fan 15'].replace('Na',nan)
team3['Fan 15'] = team3['Fan 15'].replace('nA',nan)
team3['Fan 15'] = team3['Fan 15'].astype(float)
team3['Fan 15'].unique()

array([ 90.,  nan,  80.,  70., 100.,  85.,  75.,  78.,  50.,  67.,  57.,
        59.,  47.,  60.,  79.,  49.,  40.,  68.,  87.,  45.])

In [72]:
team3['Fan 22'].unique()

array(['95', '90', '85', nan, '80', '75', '70', 'Na', '65', '60', '100',
       '50', '97', '45', '55', '40', '96', '68'], dtype=object)

In [73]:
team3['Fan 22'] = team3['Fan 22'].replace('Na',nan)
team3['Fan 22'] = team3['Fan 22'].astype(float)
team3['Fan 22'].unique()

array([ 95.,  90.,  85.,  nan,  80.,  75.,  70.,  65.,  60., 100.,  50.,
        97.,  45.,  55.,  40.,  96.,  68.])

In [74]:
team3['Fan 26'].unique()

array(['97', '81', '84', '92', '88', '91', '90', '93', '100', '98', '89',
       '87', '86', '95', nan, 'NA ', '79', '82', '85', '94', '83', '96',
       '80', '73', '60', '74', '62', '56', '75', '63', '70', '76', '58',
       '46', '77', '65', '78', '72', '67', '68', '69', 'NAN', '45'],
      dtype=object)

In [75]:
team3['Fan 26'] = team3['Fan 26'].replace('NA ',nan)
team3['Fan 26'] = team3['Fan 26'].replace('NAN',nan)
team3['Fan 26'] = team3['Fan 26'].astype(float)
team3['Fan 26'].unique()

array([ 97.,  81.,  84.,  92.,  88.,  91.,  90.,  93., 100.,  98.,  89.,
        87.,  86.,  95.,  nan,  79.,  82.,  85.,  94.,  83.,  96.,  80.,
        73.,  60.,  74.,  62.,  56.,  75.,  63.,  70.,  76.,  58.,  46.,
        77.,  65.,  78.,  72.,  67.,  68.,  69.,  45.])

Now, let's explore the `'Genre'` column to see if there are invisible characters.

In [76]:
team3['Genre'].unique()

array(['Crime', 'Thriller', 'Comedy', 'Family', 'Romance', 'War',
       'Action', 'Fantasy', 'Adventure', 'Sci-fi', 'Drama', 'Musical',
       'Horror', 'Suspense', 'Mystery ', 'Acction', 'Comedy ', 'Action ',
       'Science Fiction', 'Terror', 'Mystery'], dtype=object)

`Science Fiction` and `Sci-Fi` are the same category. Also, we'll correct spelling mistakes and remove trailing spaces.

In [77]:
team3['Genre'] = team3['Genre'].replace('Science Fiction','Sci-Fi')
team3['Genre'] = team3['Genre'].replace('Sci-fi','Sci-Fi')
team3['Genre'] = team3['Genre'].str.replace(' ','')
team3['Genre'] = team3['Genre'].replace('Acction','Action')
team3['Genre'].unique()

array(['Crime', 'Thriller', 'Comedy', 'Family', 'Romance', 'War',
       'Action', 'Fantasy', 'Adventure', 'Sci-Fi', 'Drama', 'Musical',
       'Horror', 'Suspense', 'Mystery', 'Terror'], dtype=object)

Our data is clean, with the correct data types.

In [78]:
team3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 180 entries, Pulp Fiction (1994)   to Star Wars: The Force Awakens (2015)
Data columns (total 41 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Year                          180 non-null    int64  
 1   Runtime (min)                 180 non-null    int64  
 2   Genre                         180 non-null    object 
 3   Budget (Million USD)          180 non-null    float64
 4   USA Revenue (Million USD)     179 non-null    float64
 5   Global Revenue (Million USD)  180 non-null    float64
 6   IMDb                          180 non-null    int64  
 7   Rotten Tomatoes               180 non-null    int64  
 8   Metacritic                    180 non-null    int64  
 9   Filmaffinity                  179 non-null    float64
 10  Personal                      161 non-null    float64
 11  Fans Average                  180 non-null    float64
 12  Fan 1            

Now, let's verify that the rating columns have the correct scale

In [79]:
team3.iloc[:,6:11].describe()

,IMDb,Rotten Tomatoes,Metacritic,Filmaffinity,Personal
count,180.000000,180.000000,180.000000,179.000000,161.000000
mean,74.161111,79.283333,70.583333,65.710615,87.068323
std,8.246322,17.070107,14.335515,10.350446,9.129707
min,51.000000,23.000000,34.000000,7.200000,60.000000
25%,68.750000,70.000000,60.000000,59.000000,80.000000
50%,75.000000,85.500000,72.000000,66.000000,90.000000
75%,80.000000,92.250000,81.000000,72.000000,94.000000
max,93.000000,100.000000,97.000000,88.000000,100.000000


In [80]:
team3.iloc[:,11:25].describe()

,Fans Average,Fan 1,Fan 2,Fan 3,Fan 4,Fan 5,Fan 6,Fan 7,Fan 8,Fan 9,Fan 10,Fan 11,Fan 12,Fan 13
count,180.000000,120.000000,158.00000,113.000000,110.000000,122.000000,112.00000,71.000000,94.000000,164.000000,88.000000,71.000000,180.000000,133.000000
mean,82.747222,93.991667,81.93038,87.424779,97.409091,93.770492,87.81250,87.464789,89.468085,75.713415,95.022727,78.098592,78.477778,89.263158
std,6.301267,9.744204,11.70830,9.938027,9.928455,12.099988,14.92898,10.102095,10.586766,10.173806,5.152786,22.890519,12.426974,5.793014
min,57.200000,50.000000,40.00000,9.000000,10.000000,0.000000,30.00000,70.000000,50.000000,50.000000,85.000000,0.000000,44.000000,70.000000
25%,79.100000,90.000000,76.00000,90.000000,100.000000,90.000000,80.00000,80.000000,90.000000,70.000000,90.000000,73.000000,70.000000,86.000000
50%,84.200000,100.000000,83.00000,90.000000,100.000000,100.000000,95.00000,90.000000,90.000000,77.000000,96.000000,85.000000,82.000000,90.000000
75%,87.400000,100.000000,90.00000,90.000000,100.000000,100.000000,100.00000,100.000000,98.750000,80.000000,100.000000,94.500000,88.000000,94.000000
max,93.200000,100.000000,100.00000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000,100.000000,100.000000,100.000000,96.000000,100.000000


In [81]:
team3.iloc[:,25:].describe()

,Fan 14,Fan 15,Fan 16,Fan 17,Fan 18,Fan 19,Fan 20,Fan 21,Fan 22,Fan 23,Fan 24,Fan 25,Fan 26,Fan 27,Fan 28,Fan 29
count,154.000000,124.000000,180.000000,114.000000,147.000000,176.0,116.000000,116.000000,118.000000,176.000000,174.000000,81.000000,147.000000,71.000000,180.000000,179.000000
mean,81.538961,75.225806,90.150000,85.982456,86.891156,100.0,93.913793,93.913793,74.457627,65.306818,46.919540,90.987654,81.591837,64.718310,77.066667,90.195531
std,5.977226,12.427244,4.472729,10.128364,19.291963,0.0,8.909759,8.909759,15.466890,25.670376,25.906331,9.950495,10.144676,10.918115,30.478786,13.443213
min,64.000000,40.000000,70.000000,60.000000,20.000000,100.0,60.000000,60.000000,40.000000,0.000000,0.000000,70.000000,45.000000,20.000000,0.000000,50.000000
25%,78.000000,70.000000,88.000000,80.000000,80.000000,100.0,90.000000,90.000000,61.250000,50.000000,30.000000,80.000000,77.500000,60.000000,60.000000,80.000000
50%,82.000000,78.000000,90.000000,90.000000,100.000000,100.0,100.000000,100.000000,75.000000,75.000000,50.000000,90.000000,83.000000,65.000000,90.000000,100.000000
75%,85.000000,80.000000,94.000000,95.000000,100.000000,100.0,100.000000,100.000000,90.000000,85.250000,70.000000,100.000000,88.500000,70.000000,100.000000,100.000000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.0,100.000000,100.000000,100.000000,100.000000,95.000000,100.000000,100.000000,85.000000,100.000000,100.000000


In [82]:
team3.to_csv('../datasets/team3_clean.csv')

### Team 4

The first order of business is loading the dataset into a `pandas DataFrame` object.

In [83]:
team4 = pd.read_csv('../datasets/team4.csv')

Let’s explore our database so we can get a glimpse at the data

In [84]:
team4.head()

,Movie,Duration (in minutes),Budget (in millions of dollars),Sales (in millions of dollars),Genre,IMDb Rating,Rotten Tomatoes,PopcornMeter,Metacritic Rating,Personal Rating,...,Person_11,Person_12,Person_13,Person_14,Person_15,Person_16,Person_17,Person_18,Person_19,Person_20
0,Inception,160.0,148.0,836.8,Action,88,87,91,74,85.0,...,79.0,86.0,97,81.0,79.0,92.0,79.0,95.0,92.0,81.0
1,The Dark Knight,185.0,152.0,1004.9,Action,90,94,94,84,95.0,...,91.0,85.0,97,89.0,89.0,84.0,94.0,97.0,94.0,85.0
2,Interstellar,165.0,169.0,677.5,Adventure,86,76,86,74,97.0,...,84.0,89.0,77,78.0,95.0,85.0,93.0,89.0,95.0,82.0
3,The Matrix,63.0,136.0,466.3,Action,87,83,85,73,90.0,...,81.0,88.0,96,85.0,85.0,91.0,84.0,86.0,82.0,79.0
4,Forrest Gump,55.0,142.0,678.2,Drama,88,76,95,82,100.0,...,92.0,93.0,84,79.0,83.0,81.0,96.0,96.0,87.0,83.0


We can check the dimensions of our dataframe with `pd.shape`.

In [85]:
team4.shape

(199, 30)

We observe that there are 199 observations with 30 attributes.

#### Column cleaning

In this section we will clean the column names so they look homogeneous, without extra (invisible) characters.

First of all, we print a list of column names.

In [86]:
team4.columns

Index(['Movie', 'Duration (in minutes)', 'Budget (in millions of dollars)',
       'Sales (in millions of dollars)', 'Genre', 'IMDb Rating',
       'Rotten Tomatoes', 'PopcornMeter', 'Metacritic Rating',
       'Personal Rating', 'Person_1', 'Person_2', 'Person_3', 'Person_4',
       'Person_5', 'Person_6', 'Person_7', 'Person_8', 'Person_9', 'Person_10',
       'Person_11', 'Person_12', 'Person_13', 'Person_14', 'Person_15',
       'Person_16', 'Person_17', 'Person_18', 'Person_19', 'Person_20'],
      dtype='object')

We will change `Movie` to `Title` and `Duration` to `Runtime`.

In [87]:
team4.rename(columns = {'Movie':'Title', 'Duration (in minutes)':'Runtime (in minutes)'}, inplace=True)
team4.columns

Index(['Title', 'Runtime (in minutes)', 'Budget (in millions of dollars)',
       'Sales (in millions of dollars)', 'Genre', 'IMDb Rating',
       'Rotten Tomatoes', 'PopcornMeter', 'Metacritic Rating',
       'Personal Rating', 'Person_1', 'Person_2', 'Person_3', 'Person_4',
       'Person_5', 'Person_6', 'Person_7', 'Person_8', 'Person_9', 'Person_10',
       'Person_11', 'Person_12', 'Person_13', 'Person_14', 'Person_15',
       'Person_16', 'Person_17', 'Person_18', 'Person_19', 'Person_20'],
      dtype='object')

Finally, we will set the `Title` column as the indexing column of our dataframe.

In [88]:
team4.set_index('Title', inplace=True)

#### Data cleaning

We will follow the guidelines stated below for cleaning the data for each column:
- `Year`: This column will be left unchanged. Data type can be `str` or `int`.
- `Runtime`: Data type should be `int`. Remove any indication to a unit of measurement.
- `Genre`: Data type should be `str`. We will capitalize the data, and if there is more than one genre, we will keep the first one and remove the rest.
- `Budget`: Data type can be `int` or `float`. To that end, we will remove dollar signs as well as the thousands separator. We will scale the data so the unit of measurment is millions of dollars.
- `Revenue`: Same treatment as that of `Budget`.
- Rating columns: Data type can be `int` or `float`. Change scale to [0,100].

Let's explore the type of each of the columns. In the table below we can also see if there is missing data.

In [89]:
team4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 199 entries, Inception to Keeping Up with the Joneses
Data columns (total 29 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Runtime (in minutes)             199 non-null    float64
 1   Budget (in millions of dollars)  199 non-null    float64
 2   Sales (in millions of dollars)   199 non-null    object 
 3   Genre                            199 non-null    object 
 4   IMDb Rating                      199 non-null    int64  
 5   Rotten Tomatoes                  199 non-null    int64  
 6   PopcornMeter                     199 non-null    int64  
 7   Metacritic Rating                199 non-null    int64  
 8   Personal Rating                  190 non-null    float64
 9   Person_1                         198 non-null    float64
 10  Person_2                         198 non-null    float64
 11  Person_3                         195 non-null    float64


We observe that columns `Sales`, and `Person 4, 13` have the wrong type. Let's explore each column in turn, and convert them to `int` or `float` as appropriate.

In [90]:
team4['Sales (in millions of dollars)'].unique()

array(['836.8', '1004.9', '677.5', '466.3', '678.2', '250.3', '213.9',
       '101.2', '73.3', '457.6', '1142.5', '775.4', '1033.9', '2202.2',
       '2925.5', '1518.8', '1001.8', '968.5', '394.4', '388.8', '78.3',
       '106.3', '471.6', '474.2', '482.3', '322.2', '272.7', '213.2',
       '46.8', '291.5', '406.9', '133.4', '24.1', '271.4', '105.7',
       '110.2', '90.3', '66', '26.5', '224.9', '316.8', '173', '49.5',
       '263.1', '447.4', '109.7', '375.2', '48.3', '259.3', '526.9',
       '520.9', '141.5', '401.5', '225', '200.1', '125', '300', '119',
       '214', '276', '1040.1', '656', '849.5', '485.7', '2058', '1325.6',
       '1077', '1055.1', '155', '348', '312', '783', '141', '850', '213',
       '1,346', '2,003', '773', '791', '86', '53', '350', '1,515', '533',
       '226', '1,110', '376', '414', '263', '192', '58', '456', '102',
       '137', '293', '42', '15', '165', '542', '20', '304', '110', '539',
       '1051', '346', '211', '160', '1266', '51.6', '758', '1280', '1

For the `Sales` column we need to remove the thousands separator and then convert the column to `float`.

In [91]:
team4['Sales (in millions of dollars)'] = team4['Sales (in millions of dollars)'].str.replace(',','')
team4['Sales (in millions of dollars)'] = team4['Sales (in millions of dollars)'].astype(float)
team4['Sales (in millions of dollars)'].unique()

array([ 836.8, 1004.9,  677.5,  466.3,  678.2,  250.3,  213.9,  101.2,
         73.3,  457.6, 1142.5,  775.4, 1033.9, 2202.2, 2925.5, 1518.8,
       1001.8,  968.5,  394.4,  388.8,   78.3,  106.3,  471.6,  474.2,
        482.3,  322.2,  272.7,  213.2,   46.8,  291.5,  406.9,  133.4,
         24.1,  271.4,  105.7,  110.2,   90.3,   66. ,   26.5,  224.9,
        316.8,  173. ,   49.5,  263.1,  447.4,  109.7,  375.2,   48.3,
        259.3,  526.9,  520.9,  141.5,  401.5,  225. ,  200.1,  125. ,
        300. ,  119. ,  214. ,  276. , 1040.1,  656. ,  849.5,  485.7,
       2058. , 1325.6, 1077. , 1055.1,  155. ,  348. ,  312. ,  783. ,
        141. ,  850. ,  213. , 1346. , 2003. ,  773. ,  791. ,   86. ,
         53. ,  350. , 1515. ,  533. ,  226. , 1110. ,  376. ,  414. ,
        263. ,  192. ,   58. ,  456. ,  102. ,  137. ,  293. ,   42. ,
         15. ,  165. ,  542. ,   20. ,  304. ,  110. ,  539. , 1051. ,
        346. ,  211. ,  160. , 1266. ,   51.6,  758. , 1280. , 1453. ,
      

In [92]:
team4['Person_4'].unique()

array(['86', '83', '91', '94', '81', '89', '88', '79', '72', '77', '71',
       '84', '70', '87', '82', '74', '95', '90', '75', '100', '98', '73',
       '96', '85', '80', '92', ' ', '59', '56', '60', '99', '64', '62',
       '97', nan, '55', '67', '54', '68', '66', '9', '50', '93', '29',
       '48', '65', '13', '43'], dtype=object)

For the `Person_4` column we need to change an blank cell to `nan` and then convert the column to `float`.

In [93]:
team4['Person_4'] = team4['Person_4'].replace(' ',nan)
team4['Person_4'] = team4['Person_4'].astype(float)
team4['Person_4'].unique()

array([ 86.,  83.,  91.,  94.,  81.,  89.,  88.,  79.,  72.,  77.,  71.,
        84.,  70.,  87.,  82.,  74.,  95.,  90.,  75., 100.,  98.,  73.,
        96.,  85.,  80.,  92.,  nan,  59.,  56.,  60.,  99.,  64.,  62.,
        97.,  55.,  67.,  54.,  68.,  66.,   9.,  50.,  93.,  29.,  48.,
        65.,  13.,  43.])

In [94]:
team4['Person_13'].unique()

array(['97', '77', '96', '84', '92', '88', '85', '89', '91', '75', '72',
       '79', '94', '82', '81', '83', '76', '99', '78', '90', '87', '86',
       '95', '93', '54', '71', '57', '59', '56', '60', '98', '67', '50',
       '63', '55', '80', nan, '70', '64', '10', '100', '40', '9', '8',
       '73', ' ', '66'], dtype=object)

Similarly, for the `Person_15` column we also need to change an blank cell to `nan` and then convert the column to `float`.

In [95]:
team4['Person_13'] = team4['Person_13'].replace(' ',nan)
team4['Person_13'] = team4['Person_13'].astype(float)
team4['Person_13'].unique()

array([ 97.,  77.,  96.,  84.,  92.,  88.,  85.,  89.,  91.,  75.,  72.,
        79.,  94.,  82.,  81.,  83.,  76.,  99.,  78.,  90.,  87.,  86.,
        95.,  93.,  54.,  71.,  57.,  59.,  56.,  60.,  98.,  67.,  50.,
        63.,  55.,  80.,  nan,  70.,  64.,  10., 100.,  40.,   9.,   8.,
        73.,  66.])

Now, let's explore the `'Genre'` column to see if there are invisible characters.

In [96]:
team4['Genre'].unique()

array(['Action', 'Adventure', 'Drama', 'Crime', 'Animation', 'Horror',
       'Biography', 'Comedy', 'Drama ', 'Action ', 'Romance'],
      dtype=object)

We need to remove trailing spaces.

In [97]:
team4['Genre'] = team4['Genre'].str.replace(' ','')
team4['Genre'].unique()

array(['Action', 'Adventure', 'Drama', 'Crime', 'Animation', 'Horror',
       'Biography', 'Comedy', 'Romance'], dtype=object)

Our data is clean, with the correct data types.

In [98]:
team4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 199 entries, Inception to Keeping Up with the Joneses
Data columns (total 29 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Runtime (in minutes)             199 non-null    float64
 1   Budget (in millions of dollars)  199 non-null    float64
 2   Sales (in millions of dollars)   199 non-null    float64
 3   Genre                            199 non-null    object 
 4   IMDb Rating                      199 non-null    int64  
 5   Rotten Tomatoes                  199 non-null    int64  
 6   PopcornMeter                     199 non-null    int64  
 7   Metacritic Rating                199 non-null    int64  
 8   Personal Rating                  190 non-null    float64
 9   Person_1                         198 non-null    float64
 10  Person_2                         198 non-null    float64
 11  Person_3                         195 non-null    float64


Now, let's verify that the rating columns have the correct scale

In [99]:
team4.iloc[:,4:8].describe()

,IMDb Rating,Rotten Tomatoes,PopcornMeter,Metacritic Rating
count,199.000000,199.000000,199.000000,199.000000
mean,73.572864,75.562814,79.010050,67.341709
std,8.994581,20.908044,14.251702,16.661024
min,40.000000,1.000000,31.000000,0.000000
25%,68.500000,65.000000,70.500000,57.000000
50%,74.000000,81.000000,82.000000,68.000000
75%,80.000000,91.500000,90.000000,80.000000
max,93.000000,100.000000,98.000000,100.000000


In [100]:
team4.iloc[:,8:18].describe()

,Personal Rating,Person_1,Person_2,Person_3,Person_4,Person_5,Person_6,Person_7,Person_8,Person_9
count,190.000000,198.000000,198.000000,195.000000,197.000000,197.000000,195.000000,196.000000,194.000000,195.000000
mean,77.273684,79.272727,79.858586,79.225641,78.040609,76.598985,77.600000,78.688776,81.221649,80.071795
std,14.842340,18.630970,14.707303,15.672892,15.700916,18.297613,17.762871,17.735668,15.585211,14.977474
min,20.000000,1.000000,7.000000,1.000000,9.000000,1.000000,0.000000,0.000000,10.000000,20.000000
25%,70.000000,77.000000,72.000000,75.000000,70.000000,70.000000,68.500000,71.000000,77.000000,70.000000
50%,80.000000,84.000000,80.500000,80.000000,81.000000,80.000000,81.000000,82.000000,84.000000,81.000000
75%,89.000000,90.000000,90.000000,90.000000,89.000000,89.000000,90.000000,90.000000,90.000000,91.500000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [101]:
team4.iloc[:,18:].describe()

,Person_10,Person_11,Person_12,Person_13,Person_14,Person_15,Person_16,Person_17,Person_18,Person_19,Person_20
count,198.000000,195.000000,193.000000,195.000000,193.000000,194.000000,195.000000,198.000000,195.000000,191.000000,197.000000
mean,80.525253,80.958974,78.797927,80.230769,79.362694,80.783505,80.174359,79.308081,77.830769,83.125654,80.360406
std,17.531027,15.252458,18.534753,16.580111,16.674677,15.883466,16.621014,15.723210,16.473255,14.025725,15.118142
min,0.000000,8.000000,1.000000,8.000000,1.000000,10.000000,1.000000,1.000000,8.000000,8.000000,8.000000
25%,77.250000,73.000000,70.000000,72.000000,70.000000,76.000000,74.500000,72.000000,67.000000,79.500000,75.000000
50%,82.000000,83.000000,84.000000,84.000000,80.000000,84.000000,84.000000,82.000000,81.000000,86.000000,81.000000
75%,90.000000,90.000000,92.000000,92.000000,92.000000,91.000000,90.000000,89.000000,90.000000,92.000000,90.000000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [102]:
team4.to_csv('../datasets/team4_clean.csv')

### Team 5

The first order of business is loading the dataset into a `pandas DataFrame` object.

In [103]:
team5 = pd.read_csv('../datasets/team5.csv')

Let’s explore our database so we can get a glimpse at the data

In [104]:
team5.head()

,No.,Name,Year,Duration,Genre,Budget (Dollars),Income (Dollars),IMDb,Rotten Tomatoes,Film Affinity,...,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35
0,1.0,At eternity's Gate,2018.0,111 min,Drama,NaN,"$11,519,666.00",69.0,79.0,62.0,...,50.0,60.0,100.0,100.0,80.0,70.0,NaN,65.0,70.0,75.0
1,2.0,Green Book,2018.0,130 min,Drama,"$23,000,000.00","$321,752,656.00",82.0,77.0,76.0,...,80.0,100.0,75.0,75.0,75.0,99.0,NaN,65.0,86.0,100.0
2,3.0,Amadeus,1984.0,160 min,Drama,"$18,000,000.00","$52,071,304.00",84.0,89.0,77.0,...,50.0,60.0,50.0,50.0,85.0,75.0,NaN,100.0,90.0,80.0
3,4.0,Die Hard,1988.0,132 min,Action,"$28,000,000.00","$143,651,650.00",82.0,94.0,72.0,...,85.0,80.0,80.0,80.0,95.0,60.0,90.0,90.0,75.0,60.0
4,5.0,Roma,2018.0,135 min,Drama,"$15,000,000.00","$5,100,000.00",77.0,96.0,70.0,...,70.0,60.0,60.0,60.0,90.0,70.0,NaN,65.0,75.0,60.0


We can check the dimensions of our dataframe with `pd.shape`.

In [105]:
team5.shape

(201, 36)

We observe that there are 201 observations with 36 attributes.

#### Column cleaning

In this section we will clean the column names so they look homogeneous, without extra (invisible) characters.

First of all, we print a list of column names.

In [106]:
team5.columns

Index(['No.', 'Name', 'Year', 'Duration', 'Genre', 'Budget (Dollars)',
       'Income (Dollars)', 'IMDb', 'Rotten Tomatoes', 'Film Affinity',
       'Metacritic', 'Matthias', 'Donovan', 'Alex', 'Jorge ', 'Gael',
       'Survey Ratings', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19',
       'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23',
       'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27',
       'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31',
       'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35'],
      dtype='object')

The first column is not necessary, so we drop it. We will capitalize all column names, rename `Name` to `Title`, and change `Unnamed:` to `Person`.

In [107]:
team5.drop(columns=['No.'], inplace=True)

In [108]:
team5_column_names = team5.columns.str.replace('Unnamed:','Person')
team5.columns = team5_column_names

In [109]:
team5.rename(columns = {'Name':'Title', 'Duration':'Runtime'}, inplace=True)
team5.columns

Index(['Title', 'Year', 'Runtime', 'Genre', 'Budget (Dollars)',
       'Income (Dollars)', 'IMDb', 'Rotten Tomatoes', 'Film Affinity',
       'Metacritic', 'Matthias', 'Donovan', 'Alex', 'Jorge ', 'Gael',
       'Survey Ratings', 'Person 17', 'Person 18', 'Person 19', 'Person 20',
       'Person 21', 'Person 22', 'Person 23', 'Person 24', 'Person 25',
       'Person 26', 'Person 27', 'Person 28', 'Person 29', 'Person 30',
       'Person 31', 'Person 32', 'Person 33', 'Person 34', 'Person 35'],
      dtype='object')

In [110]:
personal_rating_columns = team5.columns[16:]
other_columns = list(team5.columns[:16])
personal_rating_columns

Index(['Person 17', 'Person 18', 'Person 19', 'Person 20', 'Person 21',
       'Person 22', 'Person 23', 'Person 24', 'Person 25', 'Person 26',
       'Person 27', 'Person 28', 'Person 29', 'Person 30', 'Person 31',
       'Person 32', 'Person 33', 'Person 34', 'Person 35'],
      dtype='object')

In [111]:
new_personal_rating_columns = ['Person '+str(i+1) for i in range(19)]
new_personal_rating_columns

['Person 1',
 'Person 2',
 'Person 3',
 'Person 4',
 'Person 5',
 'Person 6',
 'Person 7',
 'Person 8',
 'Person 9',
 'Person 10',
 'Person 11',
 'Person 12',
 'Person 13',
 'Person 14',
 'Person 15',
 'Person 16',
 'Person 17',
 'Person 18',
 'Person 19']

In [112]:
team5.columns = other_columns + new_personal_rating_columns
team5.columns

Index(['Title', 'Year', 'Runtime', 'Genre', 'Budget (Dollars)',
       'Income (Dollars)', 'IMDb', 'Rotten Tomatoes', 'Film Affinity',
       'Metacritic', 'Matthias', 'Donovan', 'Alex', 'Jorge ', 'Gael',
       'Survey Ratings', 'Person 1', 'Person 2', 'Person 3', 'Person 4',
       'Person 5', 'Person 6', 'Person 7', 'Person 8', 'Person 9', 'Person 10',
       'Person 11', 'Person 12', 'Person 13', 'Person 14', 'Person 15',
       'Person 16', 'Person 17', 'Person 18', 'Person 19'],
      dtype='object')

Finally, we will set the `Title` column as the indexing column of our dataframe.

In [113]:
team5.set_index('Title', inplace=True)

#### Data cleaning

We will follow the guidelines stated below for cleaning the data for each column:
- `Year`: This column will be left unchanged. Data type can be `str` or `int`.
- `Runtime`: Data type should be `int`. Remove any indication to a unit of measurement.
- `Genre`: Data type should be `str`. We will capitalize the data, and if there is more than one genre, we will keep the first one and remove the rest.
- `Budget`: Data type can be `int` or `float`. To that end, we will remove dollar signs as well as the thousands separator. We will scale the data so the unit of measurment is millions of dollars.
- `Revenue`: Same treatment as that of `Budget`.
- Rating columns: Data type can be `int` or `float`. Change scale to [0,100].

Let's explore the type of each of the columns. In the table below we can also see if there is missing data.

In [114]:
team5.info()

<class 'pandas.core.frame.DataFrame'>
Index: 201 entries, At eternity's Gate to Team members: Alejandro Haro Silva, Gael Guzmán Aguila, Donovan Isay Medina Jasso, Jorge Hermosillo Padilla, Matthias Granados
Data columns (total 34 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Year              200 non-null    float64
 1   Runtime           200 non-null    object 
 2   Genre             200 non-null    object 
 3   Budget (Dollars)  186 non-null    object 
 4   Income (Dollars)  191 non-null    object 
 5   IMDb              197 non-null    float64
 6   Rotten Tomatoes   198 non-null    float64
 7   Film Affinity     200 non-null    float64
 8   Metacritic        198 non-null    float64
 9   Matthias          182 non-null    float64
 10  Donovan           177 non-null    float64
 11  Alex              177 non-null    float64
 12  Jorge             181 non-null    float64
 13  Gael              174 non-null    float64
 14  Sur

We observe that columns `Duration`, `Budget`, `Income`, `Person 4` and `Person 5` have the wrong type. Let's explore each column in turn, and convert them to `int` or `float` as appropriate.

In [115]:
team5['Runtime'].unique()

array(['111 min', '130 min', '160 min', '132 min', '135 min', '92 min',
       '104 min', '109 min', '95 min', '118 min', '86 min', '121 min',
       '108 min', '122 min', '119 min', '97 min', '169 min', '107 min',
       '91 min', '96 min', '116 min', '100 min', '106 min', '105 min',
       '120 min', '209 min', '117 min', '125 min', '124 min', '93 min',
       '98 min', '113 min', '126 min', '123 min', '139 min', '142 min',
       '180 min', '94 min', '137 min', '101 min', '112 min', '164 min',
       '121min', '110 min', '88 min', '101min', '102min', '129min',
       '148min', '96min', '115min', '92min', '81min', '90min', '93min',
       '106min', '100min', '143min', '108min', '94min', '145min',
       '169min', '111min', '194min', '175min', '179min', '144min',
       '95min', '136min', '140min', '98min', '181min', '149min', '147min',
       '134min', '117min', '124min', '130min', '138min', '122min',
       '131min', '110min', '133min', '163min', '126min', '158min',
       '128min',

We need to remove the string `'min'` and the trailing spaces.

In [116]:
team5['Runtime'] = team5['Runtime'].str.replace('min','')
team5['Runtime'] = team5['Runtime'].str.replace(' ','')
team5['Runtime'] = team5['Runtime'].astype(float)
team5['Runtime'].unique()

array([111., 130., 160., 132., 135.,  92., 104., 109.,  95., 118.,  86.,
       121., 108., 122., 119.,  97., 169., 107.,  91.,  96., 116., 100.,
       106., 105., 120., 209., 117., 125., 124.,  93.,  98., 113., 126.,
       123., 139., 142., 180.,  94., 137., 101., 112., 164., 110.,  88.,
       102., 129., 148., 115.,  81.,  90., 143., 145., 194., 175., 179.,
       144., 136., 140., 181., 149., 147., 134., 138., 131., 133., 163.,
       158., 128., 141., 127., 242., 176., 155., 166., 170., 202., 162.,
       114., 195., 156., 192., 103., 154., 151., 168.,  nan])

In [117]:
team5['Budget (Dollars)'].unique()

array([nan, '$23,000,000.00', '$18,000,000.00', '$28,000,000.00',
       '$15,000,000.00', '$45,000,000.00', '$5,000,000.00',
       '$13,000,000.00', '$175,000,000.00', '$65,000,000.00',
       '$5,300,000.00', '$150,000,000.00', '$11,000,000.00',
       '$123,000,000.00', '$95,000,000.00', '$180,000,000.00',
       '$20,000,000.00', '$9,000,000.00', '$85,000,000.00',
       '$74,000,000.00', '$200,000,000.00', '$30,000,000.00',
       '$22,000,000.00', '$3,300,000.00', '$159,000,000.00',
       '$165,000,000.00', '$24,500,000.00', '$32,000,000.00',
       '$40,000,000.00', '$300,000,000.00', '$107,825,862.00',
       '$839,727.00', '$140,000,000.00', '$110,000,000.00',
       '$130,000,000.00', '$35,000,000.00', '$63,000,000.00',
       '$25,000,000.00', '$100,000,000.00', '$24,000,000.00',
       '$58,000,000.00', '$38,000,000.00', '$127,000,000.00',
       '$19,000,000.00', '$120,000,000.00', '$50,000,000.00',
       '$5,400,000.00', '$230,000,000.00', '$139,000,000.00',
       '$1

We notice that the amounts are given in dollars, rather than in millions of dollars as requested. Thus, we need to rescale the column by dividing it by $10^{6}$. But before that, we need to remove the dollar signs, the thousands separators, and convert to `float`. We will also rename the column to reflect the change inscale.

In [118]:
team5['Budget (Dollars)'] = team5['Budget (Dollars)'].astype(str)
team5['Budget (Dollars)'] = team5['Budget (Dollars)'].str.replace('$','')
team5['Budget (Dollars)'] = team5['Budget (Dollars)'].str.replace(',','')
team5['Budget (Dollars)'] = team5['Budget (Dollars)'].str.replace(' ','')
team5['Budget (Dollars)'] = team5['Budget (Dollars)'].replace('N/A','')
team5['Budget (Dollars)'] = team5['Budget (Dollars)'].astype(float)
team5['Budget (Dollars)'].unique()

array([           nan, 2.30000000e+07, 1.80000000e+07, 2.80000000e+07,
       1.50000000e+07, 4.50000000e+07, 5.00000000e+06, 1.30000000e+07,
       1.75000000e+08, 6.50000000e+07, 5.30000000e+06, 1.50000000e+08,
       1.10000000e+07, 1.23000000e+08, 9.50000000e+07, 1.80000000e+08,
       2.00000000e+07, 9.00000000e+06, 8.50000000e+07, 7.40000000e+07,
       2.00000000e+08, 3.00000000e+07, 2.20000000e+07, 3.30000000e+06,
       1.59000000e+08, 1.65000000e+08, 2.45000000e+07, 3.20000000e+07,
       4.00000000e+07, 3.00000000e+08, 1.07825862e+08, 8.39727000e+05,
       1.40000000e+08, 1.10000000e+08, 1.30000000e+08, 3.50000000e+07,
       6.30000000e+07, 2.50000000e+07, 1.00000000e+08, 2.40000000e+07,
       5.80000000e+07, 3.80000000e+07, 1.27000000e+08, 1.90000000e+07,
       1.20000000e+08, 5.00000000e+07, 5.40000000e+06, 2.30000000e+08,
       1.39000000e+08, 1.70000000e+08, 1.50000000e+06, 8.00000000e+07,
       6.00000000e+07, 1.15000000e+08, 9.40000000e+07, 2.20000000e+08,
      

In [119]:
team5['Budget (Dollars)'] = team5['Budget (Dollars)']/1000000
team5.rename(columns = {'Budget (Dollars)':'Budget (Million Dollars)'}, inplace=True)
team5['Budget (Million Dollars)'].unique()

array([       nan,  23.      ,  18.      ,  28.      ,  15.      ,
        45.      ,   5.      ,  13.      , 175.      ,  65.      ,
         5.3     , 150.      ,  11.      , 123.      ,  95.      ,
       180.      ,  20.      ,   9.      ,  85.      ,  74.      ,
       200.      ,  30.      ,  22.      ,   3.3     , 159.      ,
       165.      ,  24.5     ,  32.      ,  40.      , 300.      ,
       107.825862,   0.839727, 140.      , 110.      , 130.      ,
        35.      ,  63.      ,  25.      , 100.      ,  24.      ,
        58.      ,  38.      , 127.      ,  19.      , 120.      ,
        50.      ,   5.4     , 230.      , 139.      , 170.      ,
         1.5     ,  80.      ,  60.      , 115.      ,  94.      ,
       220.      ,  92.      ,  12.5     , 160.      ,   4.      ,
         6.      ,  65.4867  ,  69.      , 356.      , 321.      ,
       250.      ,  75.      , 145.      , 178.      , 290.      ,
       185.      ,  32.5     , 113.      , 210.      , 350.   

In [120]:
team5['Income (Dollars)'].unique()

array(['$11,519,666.00', '$321,752,656.00', '$52,071,304.00',
       '$143,651,650.00', '$5,100,000.00', '$979,046,652.00', '$4,100.00',
       '$74,072,344.00', '$859,076,254.00', '$136,853,506.00',
       '$57,881,056.00', '$79,316,957.00', '$1,025,521,690.00',
       '$100,502,638.00', '$119,418,501.00', '$346,654,179.00',
       '$390,411,425.00', '$759,853,685.00', '$96,942,115.00',
       '$21,061,339.00', '$643,332,467.00', '$358,375,603.00',
       '$1,159,457,500.00', '$1,029,266,990.00', '$735,102,136.00',
       '$50,193,264.00', '$205,035,819.00', '$20,710,000.00',
       '$49,449,489.00', '$814,641,172.00', '$35,899,485.00',
       '$968,853.00', '$381,441.00', '$318,321,815.00', '$17,503,446.00',
       '$108,902,486.00', '$3,987,502.00', '$5,704,857.00', '$243,711.00',
       '$661,326,987.00', '$226,945,087.00', '$775,398,007.00',
       '$1,585,634.00', '$585,796,247.00', nan, '$579,446,407.00',
       '$321,885,765.00', '$117,625,455.00', '$1,151,442,928.00',
       '

We give the `Income` column the same treatment as that of `Budget`.

In [121]:
team5['Income (Dollars)'] = team5['Income (Dollars)'].str.replace('$','')
team5['Income (Dollars)'] = team5['Income (Dollars)'].str.replace(',','')
team5['Income (Dollars)'] = team5['Income (Dollars)'].astype(float)
team5['Income (Dollars)'].unique()

array([1.15196660e+07, 3.21752656e+08, 5.20713040e+07, 1.43651650e+08,
       5.10000000e+06, 9.79046652e+08, 4.10000000e+03, 7.40723440e+07,
       8.59076254e+08, 1.36853506e+08, 5.78810560e+07, 7.93169570e+07,
       1.02552169e+09, 1.00502638e+08, 1.19418501e+08, 3.46654179e+08,
       3.90411425e+08, 7.59853685e+08, 9.69421150e+07, 2.10613390e+07,
       6.43332467e+08, 3.58375603e+08, 1.15945750e+09, 1.02926699e+09,
       7.35102136e+08, 5.01932640e+07, 2.05035819e+08, 2.07100000e+07,
       4.94494890e+07, 8.14641172e+08, 3.58994850e+07, 9.68853000e+05,
       3.81441000e+05, 3.18321815e+08, 1.75034460e+07, 1.08902486e+08,
       3.98750200e+06, 5.70485700e+06, 2.43711000e+05, 6.61326987e+08,
       2.26945087e+08, 7.75398007e+08, 1.58563400e+06, 5.85796247e+08,
                  nan, 5.79446407e+08, 3.21885765e+08, 1.17625455e+08,
       1.15144293e+09, 1.59468292e+08, 2.54088600e+08, 1.00853753e+08,
       2.83000000e+07, 9.75134850e+08, 2.36000000e+08, 1.18722608e+08,
      

In [122]:
team5['Income (Dollars)'] = team5['Income (Dollars)']/1000000
team5.rename(columns = {'Income (Dollars)':'Income (Million Dollars)'}, inplace=True)
team5['Income (Million Dollars)'].unique()

array([1.15196660e+01, 3.21752656e+02, 5.20713040e+01, 1.43651650e+02,
       5.10000000e+00, 9.79046652e+02, 4.10000000e-03, 7.40723440e+01,
       8.59076254e+02, 1.36853506e+02, 5.78810560e+01, 7.93169570e+01,
       1.02552169e+03, 1.00502638e+02, 1.19418501e+02, 3.46654179e+02,
       3.90411425e+02, 7.59853685e+02, 9.69421150e+01, 2.10613390e+01,
       6.43332467e+02, 3.58375603e+02, 1.15945750e+03, 1.02926699e+03,
       7.35102136e+02, 5.01932640e+01, 2.05035819e+02, 2.07100000e+01,
       4.94494890e+01, 8.14641172e+02, 3.58994850e+01, 9.68853000e-01,
       3.81441000e-01, 3.18321815e+02, 1.75034460e+01, 1.08902486e+02,
       3.98750200e+00, 5.70485700e+00, 2.43711000e-01, 6.61326987e+02,
       2.26945087e+02, 7.75398007e+02, 1.58563400e+00, 5.85796247e+02,
                  nan, 5.79446407e+02, 3.21885765e+02, 1.17625455e+02,
       1.15144293e+03, 1.59468292e+02, 2.54088600e+02, 1.00853753e+02,
       2.83000000e+01, 9.75134850e+02, 2.36000000e+02, 1.18722608e+02,
      

In [123]:
team5['Person 4'].unique()

array([nan, '95', '80', 'N/aa', '90', '70', '85', '100', '82', '84', '72',
       '62', '75', '78', '79', '87', '55', '98', '65', '94', '88', '45',
       '77', '86', '63'], dtype=object)

There is an entry with a string that we can subsitute for `nan`

In [124]:
team5['Person 4'] = team5['Person 4'].replace('N/aa',nan)
team5['Person 4'] = team5['Person 4'].astype(float)
team5['Person 4'].unique()

array([ nan,  95.,  80.,  90.,  70.,  85., 100.,  82.,  84.,  72.,  62.,
        75.,  78.,  79.,  87.,  55.,  98.,  65.,  94.,  88.,  45.,  77.,
        86.,  63.])

In [125]:
team5['Person 5'].unique()

array([nan, '95', '80', '100', '60', '85', '90', '75', '1oo', '74', '88',
       '65', '70', '77', '89', '78', '76', '86', '94', '84', '50', '91',
       '51', '92', '82'], dtype=object)

There is an entry with a strange string, which must likely is a $100$.

In [126]:
team5['Person 5'] = team5['Person 5'].replace('1oo',100)
team5['Person 5'] = team5['Person 5'].astype(float)
team5['Person 5'].unique()

array([ nan,  95.,  80., 100.,  60.,  85.,  90.,  75.,  74.,  88.,  65.,
        70.,  77.,  89.,  78.,  76.,  86.,  94.,  84.,  50.,  91.,  51.,
        92.,  82.])

Now, let's explore the `'Genre'` column to see if there are invisible characters.

In [127]:
team5['Genre'].unique()

array(['Drama', 'Action', 'Coming-of-age', 'Comedy', 'Comedy-drama',
       'Adventure', 'War film', 'Fantasy', 'Romance', 'Crime', 'Sci-Fi',
       'Animation', 'Thriller', 'Biography', 'Horror', 'War', 'Family',
       'Terror', 'Mystery', 'Historical', 'Suspense', nan], dtype=object)

`War` and `War film` are the same `Genre`.

In [128]:
team5['Genre'] = team5['Genre'].replace('War film','War')
team5['Genre'].unique()

array(['Drama', 'Action', 'Coming-of-age', 'Comedy', 'Comedy-drama',
       'Adventure', 'War', 'Fantasy', 'Romance', 'Crime', 'Sci-Fi',
       'Animation', 'Thriller', 'Biography', 'Horror', 'Family', 'Terror',
       'Mystery', 'Historical', 'Suspense', nan], dtype=object)

Our data is clean, with the correct data types.

In [129]:
team5.info()

<class 'pandas.core.frame.DataFrame'>
Index: 201 entries, At eternity's Gate to Team members: Alejandro Haro Silva, Gael Guzmán Aguila, Donovan Isay Medina Jasso, Jorge Hermosillo Padilla, Matthias Granados
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Year                      200 non-null    float64
 1   Runtime                   200 non-null    float64
 2   Genre                     200 non-null    object 
 3   Budget (Million Dollars)  186 non-null    float64
 4   Income (Million Dollars)  191 non-null    float64
 5   IMDb                      197 non-null    float64
 6   Rotten Tomatoes           198 non-null    float64
 7   Film Affinity             200 non-null    float64
 8   Metacritic                198 non-null    float64
 9   Matthias                  182 non-null    float64
 10  Donovan                   177 non-null    float64
 11  Alex                      177 non-null  

Now, let's verify that the rating columns have the correct scale

In [130]:
team5.iloc[:,5:14].describe()

,IMDb,Rotten Tomatoes,Film Affinity,Metacritic,Matthias,Donovan,Alex,Jorge,Gael
count,197.000000,198.000000,200.000000,198.000000,182.000000,177.000000,177.000000,181.000000,174.000000
mean,74.588832,75.606061,70.924000,67.691919,83.000000,81.983051,81.983051,81.839779,82.241379
std,8.681935,22.354240,11.992268,15.263873,11.335662,10.536449,10.536449,9.670216,11.239860
min,52.000000,8.000000,7.800000,27.000000,40.000000,40.000000,40.000000,40.000000,40.000000
25%,69.000000,65.000000,64.000000,58.000000,77.000000,77.000000,77.000000,75.000000,78.000000
50%,76.000000,84.000000,72.000000,69.000000,85.000000,84.000000,84.000000,82.000000,85.000000
75%,81.000000,93.000000,79.000000,78.000000,90.000000,89.000000,89.000000,89.000000,90.000000
max,93.000000,100.000000,98.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [131]:
team5.iloc[:,14:24].describe()

,Survey Ratings,Person 1,Person 2,Person 3,Person 4,Person 5,Person 6,Person 7,Person 8,Person 9
count,195.000000,127.000000,142.000000,163.000000,156.000000,166.000000,193.000000,196.000000,177.000000,193.000000
mean,78.548718,79.251969,82.422535,74.785276,81.641026,82.783133,74.362694,71.035714,83.242938,79.953368
std,12.956765,13.440031,13.828393,13.318391,9.649301,11.347992,15.934209,22.684994,9.454603,13.200493
min,42.000000,30.000000,0.000000,40.000000,45.000000,50.000000,25.000000,5.000000,45.000000,48.000000
25%,75.000000,75.000000,78.000000,70.000000,77.000000,76.000000,61.000000,64.750000,77.000000,70.000000
50%,80.000000,81.000000,84.000000,80.000000,80.000000,80.000000,79.000000,78.500000,85.000000,80.000000
75%,88.000000,88.000000,90.000000,84.000000,88.000000,90.000000,85.000000,86.000000,89.000000,90.000000
max,100.000000,97.000000,100.000000,95.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [132]:
team5.iloc[:,24:].describe()

,Person 10,Person 11,Person 12,Person 13,Person 14,Person 15,Person 16,Person 17,Person 18,Person 19
count,192.000000,193.000000,190.000000,193.000000,192.000000,193.000000,146.000000,191.000000,194.000000,189.000000
mean,79.447917,77.683938,75.821053,75.637306,83.265625,77.347150,81.390411,80.769634,81.226804,80.243386
std,14.514387,12.184716,14.948574,14.786785,9.670339,10.585402,11.735224,11.912811,9.663783,15.089239
min,45.000000,42.000000,20.000000,20.000000,50.000000,50.000000,10.000000,52.000000,50.000000,30.000000
25%,70.000000,70.000000,70.000000,70.000000,80.000000,70.000000,79.000000,70.000000,75.000000,75.000000
50%,84.000000,80.000000,80.000000,79.000000,84.000000,79.000000,84.000000,84.000000,80.000000,80.000000
75%,89.000000,86.000000,85.000000,84.000000,90.000000,85.000000,88.000000,89.000000,87.000000,90.000000
max,100.000000,100.000000,100.000000,100.000000,100.000000,99.000000,100.000000,100.000000,100.000000,100.000000


In [133]:
team5.to_csv('../datasets/team5_clean.csv')

### Team 6

The first order of business is loading the dataset into a `pandas DataFrame` object.

In [134]:
team6 = pd.read_csv('../datasets/team6.csv')

Let’s explore our database so we can get a glimpse at the data

In [135]:
team6.head()

,Position,Title,Runtime (mins),Year,Genre,Budget (MM),Income (MM),IMDb,Rotten tomatoes,Metacritic,...,P11,P12,P13,P14,P15,P16,P17,P18,P19,P20
0,1,Pulp Fiction,154,1994,Crime,8.0,213,89,92.0,94.0,...,80.0,NaN,90.0,90,65.0,NaN,75.0,NaN,95,90.0
1,2,Saving Private Ryan,169,1998,Drama,98.0,482,86,93.0,86.0,...,100.0,90.0,87.0,100,100.0,NaN,90.0,80.0,90,95.0
2,3,The Departed,151,2006,Crime,90.0,291,85,91.0,82.0,...,80.0,80.0,NaN,NaN,60.0,NaN,85.0,NaN,NaN,NaN
3,4,Heat,170,1995,Action,60.0,187,83,87.0,76.0,...,75.0,NaN,92.0,95,90.0,NaN,90.0,NaN,NaN,85.0
4,5,Parasite,132,2019,Thriller,11.0,263,85,98.0,96.0,...,75.0,50.0,97.0,NaN,100.0,NaN,75.0,NaN,NaN,95.0


We can check the dimensions of our dataframe with `pd.shape`.

In [136]:
team6.shape

(200, 36)

We observe that there are 200 observations with 36 attributes.

#### Column cleaning

In this section we will clean the column names so they look homogeneous, without extra (invisible) characters.

First of all, we print a list of column names.

In [137]:
team6.columns

Index(['Position', 'Title', 'Runtime (mins)', 'Year', 'Genre', 'Budget (MM)',
       'Income (MM)', 'IMDb ', 'Rotten tomatoes', 'Metacritic', 'filmaffinity',
       'M1', 'M2', 'M3', 'M4', 'M5', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7',
       'P8', 'P9', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17',
       'P18', 'P19', 'P20'],
      dtype='object')

The first column is not necessary, so we drop it. We will capitalize `Rotten tomatoes` and `filmaffinity`.

In [138]:
team6.drop(columns=['Position'], inplace=True)

In [139]:
team6.rename(columns = {'IMDb ':'IMDb','Rotten tomatoes':'Rotten Tomatoes', 'filmaffinity':'Filmaffinity'}, inplace=True)
team6.columns

Index(['Title', 'Runtime (mins)', 'Year', 'Genre', 'Budget (MM)',
       'Income (MM)', 'IMDb', 'Rotten Tomatoes', 'Metacritic', 'Filmaffinity',
       'M1', 'M2', 'M3', 'M4', 'M5', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7',
       'P8', 'P9', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17',
       'P18', 'P19', 'P20'],
      dtype='object')

Finally, we will set the `Title` column as the indexing column of our dataframe.

In [140]:
team6.set_index('Title', inplace=True)

#### Data cleaning

We will follow the guidelines stated below for cleaning the data for each column:
- `Year`: This column will be left unchanged. Data type can be `str` or `int`.
- `Runtime`: Data type should be `int`. Remove any indication to a unit of measurement.
- `Genre`: Data type should be `str`. We will capitalize the data, and if there is more than one genre, we will keep the first one and remove the rest.
- `Budget`: Data type can be `int` or `float`. To that end, we will remove dollar signs as well as the thousands separator. We will scale the data so the unit of measurment is millions of dollars.
- `Revenue`: Same treatment as that of `Budget`.
- Rating columns: Data type can be `int` or `float`. Change scale to [0,100].

Let's explore the type of each of the columns. In the table below we can also see if there is missing data.

In [141]:
team6.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, Pulp Fiction to The Devil's Rejects
Data columns (total 34 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Runtime (mins)   200 non-null    int64  
 1   Year             200 non-null    int64  
 2   Genre            200 non-null    object 
 3   Budget (MM)      200 non-null    float64
 4   Income (MM)      200 non-null    object 
 5   IMDb             200 non-null    int64  
 6   Rotten Tomatoes  199 non-null    float64
 7   Metacritic       195 non-null    float64
 8   Filmaffinity     198 non-null    float64
 9   M1               83 non-null     object 
 10  M2               69 non-null     float64
 11  M3               200 non-null    int64  
 12  M4               106 non-null    float64
 13  M5               151 non-null    float64
 14  P1               107 non-null    float64
 15  P2               96 non-null     object 
 16  P3               83 non-null     float64

We observe that columns `Income`, `M1`, `P2`, `P14`, and `P19` have the wrong type. Let's explore each column in turn, and convert them to `int` or `float` as appropriate.

In [142]:
team6['Income (MM)'].unique()

array(['213', '482', '291', '187', '263', '171', '373', '58', '193',
       '936', '897', '327', '41', '1140', '248', '1010', '181', '467',
       '466', '457', '247', '24', '2798', '84', '156', '369', '684',
       '837', '8', '2048', '101', '678', '107', '32', '316', '772', '352',
       '857', '294.8', '497.4', '246.4', '185.3', '122.9', '577.4',
       '17.6', '456.1', '247.3', '23.9', '126.2', '124.9', '128.1',
       '623.7', '185.7', '9.4', '211.6', '163.3', '243.2', '220.9',
       '32.4', '46.7', '109.7', '252.3', '47.1', '225.9', '672.8',
       '209.2', '213.2', '154', '47.3', '87.8', '262.6', '504.1', '17.5',
       '83.9', '8.3', '116.1', '40', '2257.5', '760', '385', '103', '356',
       '48', '177', '358', '30', '2800', '130', '520', '138', '31', '180',
       '34', '497', '35', '50', '28', '450', '56', '543', '214', '20',
       '456', '249', '216', '113', '22', '606', '57', '1', '212', '215',
       '61', '335', '77', '12.2', '585.2', '159.3', '654.3', '34.1',
       '

We need to remove the thousands separator.

In [143]:
team6['Income (MM)'] = team6['Income (MM)'].str.replace(',','')
team6['Income (MM)'] = team6['Income (MM)'].astype(float)
team6['Income (MM)'].unique()

array([2.1300e+02, 4.8200e+02, 2.9100e+02, 1.8700e+02, 2.6300e+02,
       1.7100e+02, 3.7300e+02, 5.8000e+01, 1.9300e+02, 9.3600e+02,
       8.9700e+02, 3.2700e+02, 4.1000e+01, 1.1400e+03, 2.4800e+02,
       1.0100e+03, 1.8100e+02, 4.6700e+02, 4.6600e+02, 4.5700e+02,
       2.4700e+02, 2.4000e+01, 2.7980e+03, 8.4000e+01, 1.5600e+02,
       3.6900e+02, 6.8400e+02, 8.3700e+02, 8.0000e+00, 2.0480e+03,
       1.0100e+02, 6.7800e+02, 1.0700e+02, 3.2000e+01, 3.1600e+02,
       7.7200e+02, 3.5200e+02, 8.5700e+02, 2.9480e+02, 4.9740e+02,
       2.4640e+02, 1.8530e+02, 1.2290e+02, 5.7740e+02, 1.7600e+01,
       4.5610e+02, 2.4730e+02, 2.3900e+01, 1.2620e+02, 1.2490e+02,
       1.2810e+02, 6.2370e+02, 1.8570e+02, 9.4000e+00, 2.1160e+02,
       1.6330e+02, 2.4320e+02, 2.2090e+02, 3.2400e+01, 4.6700e+01,
       1.0970e+02, 2.5230e+02, 4.7100e+01, 2.2590e+02, 6.7280e+02,
       2.0920e+02, 2.1320e+02, 1.5400e+02, 4.7300e+01, 8.7800e+01,
       2.6260e+02, 5.0410e+02, 1.7500e+01, 8.3900e+01, 8.3000e

In [144]:
team6['M1'].unique()

array([nan, '85', '93', '91', '92', '95', '87', '89', '94', '78', '83',
       '86', '82', '90', '96', '75', '80', '84', 'BA', '97', '76', '73',
       '79', '88', '77', '81', '74'], dtype=object)

There is a string that we can replace with `nan`

In [145]:
team6['M1'] = team6['M1'].replace('BA', nan)
team6['M1'] = team6['M1'].astype(float)
team6['M1'].unique()

array([nan, 85., 93., 91., 92., 95., 87., 89., 94., 78., 83., 86., 82.,
       90., 96., 75., 80., 84., 97., 76., 73., 79., 88., 77., 81., 74.])

We give a similar treatment to the remaining columns.

In [146]:
team6['P2'].unique()

array([nan, '80', '90', '85', '70', '100', 'NA ', '60', '72', '87', '74',
       '97', '93', '75', '83', '53', '73', '54', '64', '89', '65', '50',
       '69', '40', '57', '66', '63', '78', '55'], dtype=object)

In [147]:
team6['P2'] = team6['P2'].replace('NA ', nan)
team6['P2'] = team6['P2'].astype(float)
team6['P2'].unique()

array([ nan,  80.,  90.,  85.,  70., 100.,  60.,  72.,  87.,  74.,  97.,
        93.,  75.,  83.,  53.,  73.,  54.,  64.,  89.,  65.,  50.,  69.,
        40.,  57.,  66.,  63.,  78.,  55.])

In [148]:
team6['P14'].unique()

array(['90', '100', nan, '95', '80', 'BA', '0', '85', '50', '87', '88',
       '75', '89', '76', '70', '82', '63', '86', '84', '72', '43', '92',
       '78', '98', '32', '6', '93', '67', '65'], dtype=object)

In [149]:
team6['P14'] = team6['P14'].replace('BA', nan)
team6['P14'] = team6['P14'].astype(float)
team6['P14'].unique()

array([ 90., 100.,  nan,  95.,  80.,   0.,  85.,  50.,  87.,  88.,  75.,
        89.,  76.,  70.,  82.,  63.,  86.,  84.,  72.,  43.,  92.,  78.,
        98.,  32.,   6.,  93.,  67.,  65.])

In [150]:
team6['P19'].unique()

array(['95', '90', nan, '100', '86', '87', '96', '94', '85', '84', '91',
       '97', '83', '88', '80', '70', 'N:A', '75', '43', '67', '25', '32',
       '92', '54', '72', '65', '66', '52', '98', '78', '71', '74', '93',
       '76', '73', '63', '30', '89'], dtype=object)

In [151]:
team6['P19'] = team6['P19'].replace('N:A', nan)
team6['P19'] = team6['P19'].astype(float)
team6['P19'].unique()

array([ 95.,  90.,  nan, 100.,  86.,  87.,  96.,  94.,  85.,  84.,  91.,
        97.,  83.,  88.,  80.,  70.,  75.,  43.,  67.,  25.,  32.,  92.,
        54.,  72.,  65.,  66.,  52.,  98.,  78.,  71.,  74.,  93.,  76.,
        73.,  63.,  30.,  89.])

Now, let's explore the `'Genre'` column to see if there are invisible characters.

In [152]:
team6['Genre'].unique()

array(['Crime', 'Drama', 'Action', 'Thriller', 'Animation', 'Comedy',
       'Adventure', 'Biography', 'Mystery', 'Horror'], dtype=object)

Everything looks right. Now our data is clean, with the correct data types.

In [153]:
team6.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, Pulp Fiction to The Devil's Rejects
Data columns (total 34 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Runtime (mins)   200 non-null    int64  
 1   Year             200 non-null    int64  
 2   Genre            200 non-null    object 
 3   Budget (MM)      200 non-null    float64
 4   Income (MM)      200 non-null    float64
 5   IMDb             200 non-null    int64  
 6   Rotten Tomatoes  199 non-null    float64
 7   Metacritic       195 non-null    float64
 8   Filmaffinity     198 non-null    float64
 9   M1               82 non-null     float64
 10  M2               69 non-null     float64
 11  M3               200 non-null    int64  
 12  M4               106 non-null    float64
 13  M5               151 non-null    float64
 14  P1               107 non-null    float64
 15  P2               95 non-null     float64
 16  P3               83 non-null     float64

Now, let's verify that the rating columns have the correct scale

In [154]:
team6.iloc[:,5:14].describe()

,IMDb,Rotten Tomatoes,Metacritic,Filmaffinity,M1,M2,M3,M4,M5
count,200.000000,199.000000,195.000000,198.000000,82.000000,69.000000,200.000000,106.000000,151.000000
mean,75.370000,74.261307,64.851282,69.858586,87.036585,82.637681,69.560000,81.660377,88.264901
std,7.686253,20.585669,16.826079,11.094893,5.720755,12.857787,18.098735,18.482881,8.602869
min,53.000000,5.000000,16.000000,40.000000,73.000000,0.000000,10.000000,12.000000,72.000000
25%,70.000000,64.500000,54.000000,62.000000,84.000000,78.000000,63.750000,76.000000,81.500000
50%,76.000000,81.000000,66.000000,71.000000,87.000000,86.000000,70.000000,87.000000,89.000000
75%,81.000000,89.000000,76.000000,78.000000,92.000000,90.000000,80.000000,95.000000,96.000000
max,93.000000,100.000000,100.000000,91.000000,97.000000,96.000000,100.000000,100.000000,100.000000


In [155]:
team6.iloc[:,14:24].describe()

,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10
count,107.000000,95.000000,83.000000,119.000000,112.000000,101.000000,98.000000,114.000000,152.000000,116.000000
mean,73.196262,74.178947,75.819277,78.571429,70.446429,82.148515,79.500000,81.280702,74.618421,70.862069
std,17.210922,14.892508,16.320827,17.790005,30.372415,18.459353,16.139374,14.267054,26.547704,20.332410
min,24.000000,40.000000,10.000000,10.000000,3.000000,12.000000,12.000000,30.000000,0.000000,1.000000
25%,60.000000,60.000000,67.000000,70.000000,65.000000,70.000000,70.000000,73.000000,67.000000,58.500000
50%,75.000000,70.000000,73.000000,80.000000,77.500000,85.000000,80.000000,80.000000,79.500000,75.000000
75%,87.000000,87.000000,90.000000,92.500000,94.000000,100.000000,90.000000,92.750000,100.000000,85.000000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [156]:
team6.iloc[:,24:].describe()

,P11,P12,P13,P14,P15,P16,P17,P18,P19,P20
count,136.000000,113.000000,92.000000,107.000000,128.000000,86.000000,97.000000,73.000000,103.000000,111.000000
mean,83.102941,82.132743,83.902174,82.859813,78.296875,83.116279,84.041237,86.342466,83.048544,82.099099
std,16.127400,13.725305,15.846142,17.313670,15.300332,20.689536,16.562959,14.787588,14.617595,15.144129
min,14.000000,42.000000,16.000000,0.000000,4.000000,10.000000,0.000000,12.000000,25.000000,12.000000
25%,76.000000,72.000000,80.000000,78.000000,70.000000,71.250000,78.000000,80.000000,79.000000,74.500000
50%,85.000000,80.000000,89.000000,85.000000,80.000000,90.000000,90.000000,90.000000,85.000000,85.000000
75%,95.000000,90.000000,95.000000,95.000000,90.000000,100.000000,95.000000,100.000000,92.000000,95.000000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [157]:
team6.to_csv('../datasets/team6_clean.csv')